# Clasificación Binaria — Clase 0 (Adopción Inmediata)

> **Modelo specialist** para el metamodelo: convierte la tarea original
> multiclase de `AdoptionSpeed` (5 clases) en binaria `is_class_0` (sí/no).
>
> **Objetivo**: aportar al metamodelo una predicción focalizada en la clase rara
> (clase 0 = adopción el mismo día, solo 2.7% del dataset).
>
> **Salida**: CSV con 5 columnas para mantener compatibilidad con el metamodelo.
> - `prob_class_0`: probabilidad real predicha por este modelo.
> - `prob_class_1..4`: distribución uniforme de la masa restante `(1-p)/4`.
>
> Se reutilizan los parquets de FE generados por `04b_FeatureEngineering_Pipeline`
> (las features no cambian — solo el target).


---
## 1. Imports & Config

| Librería | Rol |
|---|---|
| `lightgbm` | Modelo de Gradient Boosting basado en árboles |
| `optuna` | Optimización bayesiana de hiperparámetros (sampler TPE) |
| `sklearn` | Split, métricas (QWK, F1, Accuracy, Balanced Accuracy) y K-Fold |
| `plotly` | Visualizaciones interactivas (barras, líneas, pie, heatmaps) |
| `joblib` | Persistencia de modelos entrenados por fold |
| `utils` | Funciones auxiliares del proyecto (`plot_confusion_matrix`) |

In [1]:
from __future__ import annotations

import os
import re
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

import lightgbm as lgb
import optuna

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from joblib import dump, load
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    f1_score,
)
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    StratifiedGroupKFold,
    train_test_split,
)

from optuna.artifacts import FileSystemArtifactStore, upload_artifact

from utils import plot_confusion_matrix, get_artifact_filename


warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path('.')
TRAIN_PATH   = NOTEBOOK_DIR / '../input/petfinder-adoption-prediction/train/train.csv'
BREED_PATH   = NOTEBOOK_DIR / '../input/petfinder-adoption-prediction/train/breed_labels.csv'
BASE_DIR = '../'

PATH_TO_MODELS           = os.path.join(BASE_DIR, 'work/models')
PATH_TO_TEMP_FILES       = os.path.join(BASE_DIR, 'work/optuna_temp_artifacts')
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, 'work/optuna_artifacts')


# ── Constants ────────────────────────────────────────────────────────────────
SEED                 = 42
AGE_CLIP_PERCENTILE  = 99
AGE_YOUNG_THRESHOLD  = 6      # months
AGE_BINS             = [0, 2, 6, 12, 24, 60, 999]
AGE_LABELS           = [0, 1, 2, 3, 4, 5]
FEE_BINS             = 4
TERNARY_YES          = 1
HEALTH_VARS          = ['Vaccinated', 'Dewormed', 'Sterilized']

# ── Splits ──────────────────────────────────────────────────────────────────
# - Si META_TRAIN_SIZE > 0: split 3-way (train + meta_train + test).
#   Útil si después vas a usar meta_train para entrenar un metamodelo encima.
# - Si META_TRAIN_SIZE = 0: split 2-way (solo train + test). USE_META_TRAIN=False.
TRAIN_SIZE       = 0.8
TEST_SIZE        = 0.2
META_TRAIN_SIZE  = 0

USE_META_TRAIN = META_TRAIN_SIZE > 0
N_FOLDS     = 5
N_TRIALS    = 100 #Codigo se probo primero con 50
NUM_CLASSES = 5
EXPERIMENT = 1
STUDY_NAME  = 'Pet Finder - Tabular LGBM Model - BinaryClass0 - Experiment ' + str(EXPERIMENT)
LABEL       = 'AdoptionSpeed'

# ── Configuración binaria ──────────────────────────────────────────────────
# La tarea original es multiclase 0..4. Acá entrenamos un modelo binario
# 'es clase 0 vs no es clase 0' para enriquecer al metamodelo con un specialist
# en la clase rara.
BINARY_TARGET_CLASS = 0
BINARY_LABEL        = f'is_class_{BINARY_TARGET_CLASS}'

COLS_TO_DROP = [
    'PetID', 'Name', 'Description',
    'Color1', 'Color2', 'Color3',
    'State', 'Breed1', 'Breed2',
    # NOTA: RescuerID se mantiene en el DF post-FE para usarlo como `groups`
    # en StratifiedGroupKFold. Se excluye de features_fe en cell 56.
]

# ── Malaysian census data (source: Dept. of Statistics Malaysia / competition solutions) ─
MALAYSIA_STATE_STATS: dict[int, dict] = {
    41324: {'population': 1_796_000, 'area_km2': 1_048,  'gdp_pc': 43_000, 'urban_pct': 0.91},  # Kuala Lumpur
    41325: {'population': 3_492_000, 'area_km2': 7_960,  'gdp_pc': 39_000, 'urban_pct': 0.95},  # Selangor
    41326: {'population': 1_624_000, 'area_km2': 1_652,  'gdp_pc': 32_000, 'urban_pct': 0.83},  # Johor
    41327: {'population': 1_813_000, 'area_km2': 14_941, 'gdp_pc': 19_000, 'urban_pct': 0.53},  # Kedah
    41328: {'population': 1_670_000, 'area_km2': 14_922, 'gdp_pc': 21_000, 'urban_pct': 0.46},  # Kelantan
    41329: {'population': 1_046_000, 'area_km2': 6_643,  'gdp_pc': 25_000, 'urban_pct': 0.66},  # Melaka
    41330: {'population': 1_036_000, 'area_km2': 6_686,  'gdp_pc': 24_000, 'urban_pct': 0.68},  # Negeri Sembilan
    41331: {'population': 1_688_000, 'area_km2': 35_965, 'gdp_pc': 41_000, 'urban_pct': 0.55},  # Pahang
    41332: {'population': 2_501_000, 'area_km2': 21_035, 'gdp_pc': 36_000, 'urban_pct': 0.75},  # Perak
    41333: {'population':   255_000, 'area_km2':    821, 'gdp_pc': 29_000, 'urban_pct': 0.82},  # Perlis
    41334: {'population': 1_679_000, 'area_km2': 1_031,  'gdp_pc': 28_000, 'urban_pct': 0.88},  # Pulau Pinang
    41335: {'population': 3_178_000, 'area_km2': 36_137, 'gdp_pc': 37_000, 'urban_pct': 0.63},  # Sarawak
    41336: {'population': 3_400_000, 'area_km2': 73_631, 'gdp_pc': 22_000, 'urban_pct': 0.54},  # Sabah
    41337: {'population': 1_210_000, 'area_km2': 7_956,  'gdp_pc': 25_000, 'urban_pct': 0.65},  # Terengganu
    41415: {'population':   100_000, 'area_km2':     94, 'gdp_pc': 48_000, 'urban_pct': 0.99},  # Putrajaya
}

np.random.seed(SEED)


# ── Feature Engineering: cachear a disco ─────────────────────────────────────
# Si PROCESS_FE=True: calcula el FE, lo guarda en disco como parquet, sigue.
# Si PROCESS_FE=False: levanta los archivos parquet pre-existentes.
# Las 3 particiones (train, meta_train, test) se guardan por separado para
# preservar los universos exactos (importante para metamodelos posteriores).
PROCESS_FE = False

FE_OUTPUT_DIR = '../input/petfinder-adoption-prediction/train/'
FE_FILES = {
    'train'     : 'train_fe.parquet',
    'meta_train': 'meta_train_fe.parquet',
    'test'      : 'test_fe.parquet',
}

print('Config OK')

c:\Users\ornec\anaconda3\envs\ldi2_pt28\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config OK


---
## 2. Problem Definition

| Propiedad | Valor |
|---|---|
| **Target** | `AdoptionSpeed` — entero ordinal 0-4 |
| **0** | Misma fecha del listado |
| **1** | 1–7 días |
| **2** | 8–30 días |
| **3** | 31–90 días |
| **4** | Sin adopción después de 100 días |
| **Task** | Clasificación multiclase ordinal (5 clases) |
| **Métrica oficial Kaggle** | **Quadratic Weighted Kappa (QWK)** — penaliza más los errores lejanos en la escala ordinal |
| **Métricas adicionales** | **Macro F1** (clase 0 es rara → F1 macro pondera por igual todas las clases); **Confusion Matrix** (detecta sesgo sistemático en clases medias); **Accuracy** (baseline de comunicación) |

### Por qué clasificación y no regresión
Aunque AdoptionSpeed es ordinal, las clases tienen semántica discreta ("mismo día" vs. "más de 100 días") y la competencia evalúa con QWK, que opera sobre clases predichas. Tratar como regresión requeriría redondear la salida y no captura correctamente la distribución de probabilidad por clase.

---
## 3. Load Data

In [2]:
df = pd.read_csv(TRAIN_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

breed_labels_df = pd.read_csv(BREED_PATH)
print(f'Shape: {breed_labels_df.shape}')
print(f'Columns: {breed_labels_df.columns.tolist()}')
breed_labels_df.head(3)

Shape: (14993, 24)
Columns: ['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed']
Shape: (307, 3)
Columns: ['BreedID', 'Type', 'BreedName']


,BreedID,Type,BreedName
0,1,1,Affenpinscher
1,2,1,Afghan Hound
2,3,1,Airedale Terrier


## 4. Split Datasets

In [3]:
# ── Separación estratificada ──────────────────────────────────────────────
# Validación: las 3 fracciones deben sumar 1.0.
assert round(TRAIN_SIZE + TEST_SIZE + META_TRAIN_SIZE, 10) == 1.0, \
    "TRAIN_SIZE + TEST_SIZE + META_TRAIN_SIZE debe sumar 1.0 (META_TRAIN_SIZE puede ser 0)"

if USE_META_TRAIN:
    # 3-way split (60/20/20 por defecto)
    train, temp = train_test_split(
        df,
        train_size=TRAIN_SIZE,
        random_state=SEED,
        stratify=df[LABEL],
    )
    relative_test_size = TEST_SIZE / (TEST_SIZE + META_TRAIN_SIZE)
    meta_train, test = train_test_split(
        temp,
        test_size=relative_test_size,
        random_state=SEED,
        stratify=temp[LABEL],
    )
    df_train      = train
    df_meta_train = meta_train
    df_test       = test
else:
    # 2-way split (solo train + test)
    df_train, df_test = train_test_split(
        df,
        train_size=TRAIN_SIZE,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=df[LABEL],
    )
    df_meta_train = None

# ── Reportes ───────────────────────────────────────────────────────────────
print(f'Modo split: {"3-way (train + meta_train + test)" if USE_META_TRAIN else "2-way (train + test)"}')
print(f'Train:      {df_train.shape}')
if USE_META_TRAIN:
    print(f'Meta train: {df_meta_train.shape}')
print(f'Test:       {df_test.shape}')

print('\nDistribución del target en train:')
print((df_train[LABEL].value_counts(normalize=True) * 100).sort_index().round(1).to_string())

if USE_META_TRAIN:
    print('\nDistribución del target en meta_train:')
    print((df_meta_train[LABEL].value_counts(normalize=True) * 100).sort_index().round(1).to_string())

print('\nDistribución del target en test:')
print((df_test[LABEL].value_counts(normalize=True) * 100).sort_index().round(1).to_string())


Modo split: 2-way (train + test)
Train:      (11994, 24)
Test:       (2999, 24)

Distribución del target en train:
AdoptionSpeed
0     2.7
1    20.6
2    26.9
3    21.7
4    28.0

Distribución del target en test:
AdoptionSpeed
0     2.7
1    20.6
2    26.9
3    21.7
4    28.0


---
## 5. Feature Engineering Functions

> **Convención**: cada función recibe y retorna un `pd.DataFrame`. Las funciones que aprenden estadísticas de los datos (frequency maps, clip values, quantile bins) aceptan esos valores como parámetros opcionales: si son `None` se calculan internamente (modo **fit** sobre train); si se pasan explícitamente se aplican directamente (modo **transform** sobre test).

### 5.1 Preprocessing Básico

**Justificación EDA:** Name tiene 8.4% de NaN — en lugar de imputar texto, la *ausencia* de nombre es información en sí misma (rescatistas menos comprometidos no nombran a las mascotas). Description tiene solo 13 NaN (0.09%) → reemplazar con string vacío para que las funciones de texto no fallen. Los códigos `0` en MaturitySize y FurLength significan "no especificado" → imputar con la moda de valores válidos.

In [4]:
def preprocess_basic(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Manejo de missings y encoding de columnas básicas.

    Parámetros
    ----------
    df_train : pd.DataFrame
        DataFrame de entrenamiento. NO debe contener AdoptionSpeed.
    df_test : pd.DataFrame
        DataFrame de prueba. NO debe contener AdoptionSpeed.
    maturity_size_mode : int, optional
        Moda de MaturitySize (excluye 0) calculada en train.
    fur_length_mode : int, optional
        Moda de FurLength (excluye 0) calculada en train.

    Retorna
    -------
    tuple[pd.DataFrame, pd.DataFrame] con columnas `has_name` agregadas y missings resueltos.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    generic_names = [
    "",
    "No Name",
    "Puppy",
    "Kittens",
    "Puppies",
    "Kitten",
    "No Name Yet",
    "Boy",
    "Unknown",
    "Girl",
    "Puppies For Adoption",
    "HUSKY",
    "6 Puppies",
    "Kitties",
    "Shih Tzu",
    "Beagle",
    "Golden",
    "Kitten For Adoption",
    "Kittens For Adoption",
    "Cute Puppies",
    "Cute Pups",
    "Nameless",
    "Unnamed",
    "2 Kittens",
    "4 Kittens",
    "Baby Girl",
    "Lovely Puppies",
    "2 Puppies",
    "3 Cute Kittens",
    "3 Kittens",
    "4 Kitten",
    "4 Puppies",
    "5 Puppies",
    "B1",
    "B2",
    "Baby Kittens",
    "Cat",
    "No Names Yet",
    "Poodle",
    "Puppy 2",
    "(No Name)",
    "2 Female Puppies",
    "3 Puppies",
    "5 Kittens",
    "5 Puppies For Adoption",
    "7 Puppies",
    "8 Puppies",
    "A1",
    "A2",
    "Adorable Pups For Adoption",
    "B",
    "Boxer",
    "Cute Kittens",
    "Cute Puppy",
    "Dog",
    "Golden Retriever",
    "Kitten 1",
    "Male Puppy",
    "Nil",
    "Noname",
    "Puppy 1",
    "Siblings",
    "Three Kittens",
    "TWO KITTENS",
    "'-",
    "2 Black Puppies",
    "2 Cute Kittens",
    "2 Cute Puppies",
    "3 Cute Pups",
    "3 Kitties",
    "3 Little Kittens",
    "3 PUPPIES FOR ADOPTION",
    "4 Cute Puppies",
    "4 Little Kitties",
    "5 Kitties",
    "A",
    "A3",
    "A4",
    "A5",
    "Adorable Kittens",
    "B3",
    "B4",
    "Black Puppy",
    "C",
    "Cat 1",
    "Cats For Adoption",
    "Cute Kitten",
    "CUTE PUPPIES FOR ADOPTION!",
    "Cute Tabby Kitten",
    "D",
    "D2",
    "Dog For Adoption",
    "F1",
    "F4",
    "Family",
    "FOR ADOPTION",
    "Found Dog",
    "Four Kittens",
    "G3",
    "Ginger Kitten",
    "Kitten 2",
    "Kitten 3",
    "Little Kitten",
    "Little Kittens",
    "Male Puppies",
    "Mixed Breed Puppies",
    "Need New Home",
    "Pup",
    "Puppy 3",
    "Puppy 4",
    "Puppy For Adoption",
    "Puppy Girl",
    "Rottweiler",
    "Schnauzer",
    "Siamese",
    "Siamese Mix Kitten",
    "Stray Puppies",
    "White Cat",
    "White Kitten",
    "'-  - 6 Lovely Puppies",
    "!",
    "!!!!",
    "!!.",
    "!.",
    "\"no Name\"",
    "#1",
    "(No Names Yet)",
    "..",
    "...",
    "[No Name]",
    "1",
    "1 Adorable Puppy",
    "1 Kitten",
    "1 Lovely Puppy For Adoption",
    "1 Persian Mixed Kitten",
    "1 Puppy For Adoption",
    "1,2,3",
    "1..2..3",
    "10 Puppies For Adoption",
    "11 Kittens",
    "12 Puppies",
    "18 Cats For NEED HOMES!!",
    "1F",
    "2",
    "2  Adorable  Kittens",
    "2  Cute Puppies",
    "2 Abandoned Puppies",
    "2 Adorable Kittens.Black& Grey",
    "2 Black Female Kittens",
    "2 Boys For Adoption",
    "2 Cute Female Puppy For Adoption",
    "2 Cute Kitten",
    "2 Cute Kittens For Adoption",
    "2 Cute Mix Breed Cats",
    "2 Female Pup",
    "2 Female Puppies(For Urgent Adoption)",
    "2 Females Spotted Brown White Puppies",
    "2 Grey Kittens-",
    "2 Kittens For Adoption",
    "2 Kittens For Adoption.",
    "2 Little Kittens",
    "2 Little Puppies",
    "2 Lovely Cats",
    "2 Lovely Healthy Kittens",
    "2 Male & 1 Female Kittens",
    "2 Playful Kittens",
    "2 Puppies For Adoption",
    "2 Puppy Male/female",
    "2 Pups ( Mix )",
    "2 Siblings (Male-Retriever Pups)",
    "2 Sister Kittens",
    "2 Stray Female Cats N 9 Kitties",
    "2 White + 1 Tabby For Adoption",
    "2 White Kittens And Calico Mama",
    "20 Little Puppies",
    "251(081016)",
    "2F",
    "3",
    "3 Abandoned Kittens",
    "3 Adorable Kittens",
    "3 Adorable Pups",
    "3 Beautiful Black Kittens",
    "3 Cats",
    "3 Cute Brown Puppies",
    "3 Cute Puppies",
    "3 Cute Puppies â¤ï¸ðâ¤ï¸",
    "3 Female Kittens",
    "3 Golden Tabby Kittens",
    "3 Kitten",
    "3 Kitten<",
    "3 Kittens For Adoption",
    "3 Kittens For Adoption (2 M/o)",
    "3 Litter Puppies",
    "3 Little Kitten",
    "3 Little Kitties",
    "3 Male 1 Female For Adoption",
    "3 Male Black Pups",
    "3 Male Kittens",
    "3 Puppies. 2 Female, 1 Male.",
    "3 Puppy",
    "3 Shih Tzu Mixed Puppy",
    "3 Siblings (Female-Retriever Pups)",
    "30 Cats For Adoption",
    "3F",
    "4 Abandoned Kittens - Need Foster",
    "4 Adorable Puppies",
    "4 Cats For Adoption",
    "4 Cute Kittens",
    "4 Cute Kittens!",
    "4 Cute Mix Breed Puppies",
    "4 Cute Poor Kittens!!!",
    "4 Female Puppies",
    "4 Healthy Playful Puppies Available",
    "4 Kitten FOR ADOPTION URGENT",
    "4 Kittens For Adoption",
    "4 Kittens Free For Adoption",
    "4 Kittens Looking For A New Home",
    "4 Kittens Need Foster",
    "4 Kittens Need Good Homes",
    "4 Little Kittens",
    "4 Little Kittens & Cat Mama",
    "4 Little Pups",
    "4 PUPPIES FOR ADOPTION",
    "4 Puppies Found",
    "4 Stray Kittens",
    "4 Stray Kittens_",
    "5",
    "5 Adorable Kittens",
    "5 Adorable Pups",
    "5 Baby Kittens",
    "5 Cute Puppies",
    "5 Cute Puppies For Adoption",
    "5 Dumped Pups",
    "5 Female Puppies Looking For A Home",
    "5 Females Puppies",
    "5 Kitten",
    "5 Kittens For Adoption",
    "5 Kittens For Forever Home",
    "5 Mixed Breed Puppies Needs HOME",
    "5 Newborn Kittens",
    "5 Puppies For Adoption - Urgent!",
    "6",
    "6 Adorable Puppies",
    "6 Cats For Adoption",
    "6 CUTE KITTENS",
    "6 Cute Puppies",
    "6 Fluffy Puppies",
    "6 Kitten",
    "6 Kittens",
    "6 New Pups",
    "6 Siblings",
    "7",
    "7 Cute Kittens For Free!",
    "7 Cute Puppies",
    "7 Puppies For Adoption",
    "7 Pups",
    "8",
    "8  Puppies (5 Males & 3 Females)",
    "8 Beautiful Puppies & Mommy",
    "8 Female Puppies",
    "9 Puppies",
    "9 Puppies + 1 Mama Dog",
    "9 Puppies For Adoption!",
    "99",
    "A (F)  B (M)",
    "A Kitten For Good Home.",
    "A Litter Of 10 Puppies",
    "A M B E R",
    "A Mom And 4 Kittens",
    "A Mum With 6 Kittens",
    "A Pair Of Kitties",
    "A Pair Of Siblings Kittens",
    "A Stray Cat",
    "A,b,c",
    "ä¸­ç§èå¿«ä¹ For ADOPTION.",
    "â¡ 5 Puppies â¡ Male & Female",
    "â¥Stray Dogsâ¥",
    "âð» Black Kittens",
    "âª The Cat Family âª",
    "A6",
    "A88",
    "A9",
    "Abandoned Dog",
    "Abandoned Kitten",
    "Abandoned Puppies",
    "Active & Lovely Pups",
    "Active Male Kitten",
    "Adoption Puppies * URGENT *",
    "Adorable Female Puppy !!",
    "Adorable For Adoption-",
    "Adorable Kitten.",
    "Adorable Kittens For Adoption (Free)",
    "ADORABLE PUPPIES FOR ADOPTION",
    "Adorable Puppy",
    "A-G",
    "B11",
    "B22",
    "B33",
    "B5",
    "Baby B&W Kitten",
    "Baby Boy",
    "Baby Dog",
    "Baby Dog And Puppies",
    "Beagle Mixed Dog",
    "Beautiful Grey Mix-Persian Kitten",
    "Beautiful Kitten",
    "Beautiful Sisters For Adoption",
    "Black & White Kitten",
    "Black & White Kittens",
    "Black Cat",
    "Black Dog",
    "Black Kitten",
    "Black Kitten Needs Home",
    "Black Mommy &1  Pup",
    "Black Puppies",
    "Black Pups",
    "Brown Kitten",
    "Brown Male Pup",
    "Brown Mommy & 2 Pups",
    "Brown Pup",
    "Brown Pups",
    "Brown Tabby Kitten Looking For Home",
    "C.C",
    "C1",
    "C137(240916)",
    "C14",
    "C1C",
    "C2",
    "C2C",
    "C3C",
    "C4C",
    "C5C",
    "C6C",
    "C7C",
    "Cat & Kitten For Adoption",
    "CAT 2 (FLUFFY)",
    "Cat A &B",
    "Cat Cat",
    "Cat D",
    "Cat E",
    "Cat For Adoption",
    "CATS",
    "Cats And  Kittens For Adoption",
    "CATS FOR ADOPTION- URGENT",
    "Cats/kittens",
    "Chihuahua",
    "Chihuahua Mix Pups",
    "Cocker Spaniel",
    "Cocker Spaniel Mummy & Puppies",
    "Cream Kitten",
    "Cute Black Kitten For Adoption",
    "Cute Black Pup Looking For A Home~",
    "Cute Black Puppy",
    "Cute Brown Pup",
    "Cute Cat",
    "Cute Cute Cute Puppies",
    "Cute Dog",
    "Cute Female Pup",
    "Cute Female Puppies Looking A Home~",
    "Cute Fluffy Kitten",
    "Cute Gray Kitten For Adoption!",
    "Cute Healthy Puppies",
    "Cute Kitten For Adopt",
    "Cute Kitten For Adoption",
    "Cute Kitten For Adoption.",
    "Cute Kitten Needs Home",
    "Cute Kittens For Adoption",
    "Cute Kittens!",
    "Cute Kitties Looking For A Home",
    "Cute Orange Kitten",
    "Cute Puppies For Adopt",
    "Cute Puppies For Adoption",
    "Cute Puppies Looking Forever Home~",
    "Cute Puppy For Adoption",
    "Cute Puppy Sisters",
    "Cute Siblings",
    "Cute Stray Cats",
    "Cute White Pup",
    "ðð 3 Kittens & A Mama ðð",
    "D117 (160415)",
    "D129(240916)",
    "D13(070216)",
    "D144(030217)",
    "D184(070214)",
    "D192(170416)",
    "D198(250415)",
    "D226(271116)",
    "D233(271016)",
    "D253(180217)",
    "D298(080217)",
    "D4",
    "D5",
    "D57(040317)",
    "D6",
    "D64(050317)",
    "D7",
    "D79(201114)",
    "D9",
    "D98(280516)",
    "Doberman",
    "Doberman  Mix Puppies",
    "Dogs",
    "E.T",
    "E.T.",
    "F10",
    "F11",
    "F2",
    "F3",
    "F5",
    "F6",
    "F7",
    "F8",
    "F9",
    "Fat Puppies",
    "Female & Her Puppies",
    "Female Dog",
    "Female Dog 1",
    "Female Dog 2",
    "Female For Adoption",
    "Female Kitten",
    "Female Kitten For Adoption",
    "Female Puppy",
    "Female Puppy 2",
    "Female Puppy 3",
    "Female Puppy 4",
    "Female Puppy 5",
    "Female Puppy 6",
    "Female Puppy For Adoption",
    "Five Cute Siblings",
    "Five Little Kitten",
    "Fluffy Orange Kitten",
    "For Adoption :)",
    "Four Adorable Kittens For Adoption!",
    "Four Adorable Newborn Kittens",
    "FREE CATS FOR ADOPTION",
    "FREE Cute Puppies For Adoption",
    "FREE KITTEN FOR ADOPT",
    "Free Kittens",
    "Free Orange Kitten For Adoption!",
    "G.O.T Puppies For Adoption",
    "G1",
    "G2",
    "German Shepherd",
    "German Shepherd Mix Puppies",
    "Ginger Kitten 2",
    "Ginger Kitten 3",
    "Ginger Kitties",
    "Girl Kitten",
    "Golden And Cream  Puppies",
    "Golden Retriever Mix Male Puppies",
    "GOOD CATS",
    "Good Home Wanted For 5 Puppies",
    "Grey Kitten",
    "H1",
    "H2",
    "H3",
    "H4",
    "H5",
    "H6",
    "H7",
    "H8",
    "Handsome Boy For Adoption",
    "Handsome Kitties",
    "Handsome Orange Cat",
    "Healthy Active Kitten",
    "Healthy Kitten!",
    "Healthy Pups",
    "In Need Of New Home~",
    "J",
    "J1",
    "J1010",
    "J1011",
    "J1012",
    "J-Female Puppy-01",
    "J-Female Puppy-02",
    "J-Male Puppy-01",
    "J-Male Puppy-02",
    "K I T T Y",
    "Kitten - Mixed Persian",
    "Kitten 001",
    "Kitten 002",
    "Kitten 01",
    "Kitten 03",
    "Kitten 1, Kitten 2",
    "Kitten A & B",
    "Kitten And Mommy Cat",
    "Kitten Cat",
    "Kitten Cat - Girl Girl",
    "Kitten F2",
    "Kitten F3",
    "KITTEN FOR ADOPT (FREE) 2",
    "KITTEN FOR ADOPTION!",
    "Kitten For Adoption! 1",
    "Kitten For Adoption! 2",
    "Kitten For Adoption! 3",
    "Kitten For Adoption.",
    "Kitten For Free",
    "Kitten Girl Girl",
    "Kitten Looking For Home",
    "Kitten N Mummy",
    "Kitten R1",
    "Kitten Rescue",
    "Kitten S1",
    "Kitten S3",
    "Kitten S5",
    "Kitten's",
    "KITTENS - URGENT ADOPTION",
    "Kittens & Mother Cat Need Homes",
    "Kittens Available For Adoption",
    "Kittens For Adoption!!!",
    "Kittens For Adoption~",
    "Kittens For Urgent Adoption",
    "Kittens Free Adoption",
    "KITTENS LITTER #2",
    "Kittens Looking For Forever Homes",
    "Kittens Need Home",
    "Kittens Need Home!!",
    "Kitties Kitties",
    "Kitties~~<3",
    "L",
    "L17",
    "Labrador  Retriever  Mix Puppies",
    "Little Gray Kitten",
    "Little Pup",
    "Little Puppies",
    "Little Puppy",
    "Little Siamese Kitten",
    "Looking For A Kitten",
    "Looking For Good Home - Puppies 1",
    "Looking For Good Home - Puppies 2",
    "Looking For Good Home For 8 Kitties",
    "Looking Newborn Puppy For Adoption",
    "Lost Dog",
    "Lovely Female Puppy",
    "Lovely Kitten",
    "Lovely Mum & Kittens",
    "M",
    "M1",
    "M1 N F1",
    "M2",
    "M3",
    "M4",
    "M5",
    "M6",
    "M8",
    "M9",
    "Male",
    "Male & Female Pup",
    "Male Kitten For Adoption",
    "Male Kittens",
    "Maltese",
    "Mama + 4 Male + 1 Female Kittens",
    "Mama And Kittens",
    "Mama Cat",
    "Mama Cat & 3 Kittens",
    "Mama Cat & 4 Babies",
    "Mama Cat + 4 Babies",
    "Mama N 5 Kitties",
    "Mini Poodle Male For Adoption",
    "Mix Breed Puppies",
    "Mix Kitten",
    "Mix Kitties Urgent Adoption",
    "Mixed Bengal For Adoption",
    "Mixed Breed Black Puppy",
    "Mixed Breed Pup",
    "Mixed Breed Puppies For Adoption.",
    "Mixed Breed Puppy",
    "Mixed Colour Kittens",
    "Mixed Mongrel Puppies (1M/2F)",
    "Mixed Puppies",
    "Mixed Puppy",
    "Mom & Her Kitties",
    "Mom Cats With 4 Kittens",
    "Mommy & 2 Kittens",
    "Mommy & 3 Kittens",
    "Mommy And 4 Kitties",
    "Mommy Cat And 3 Kittens",
    "Mongrel Puppies",
    "More Kittens For Adoption~",
    "Mother Cat",
    "Mother Dog",
    "Mummy & 3 Male Kittens",
    "Mummy + 3 Kittens",
    "Mummy And Three Kittens",
    "New Kittens",
    "Newborn Kitten",
    "No Name  Yet",
    "No Names",
    "Not Named",
    "O",
    "Orange Cats",
    "Orange Cats For Adoption",
    "Orange Kitten",
    "P",
    "P1",
    "P147(180916)",
    "P15(230916)",
    "P190(080916)",
    "P253(080916)",
    "P3",
    "P4",
    "P5",
    "P51(051116)",
    "P6",
    "P68(031016)",
    "P95(151016)",
    "Playful Kitten",
    "Playful Kittens",
    "Please Adopt Puppies",
    "Pug For Adoption",
    "Pup #1",
    "Pup 1 And 2",
    "Pup 3 & Pup 4",
    "Pup 5 & Pup 6",
    "Pup Pup Need A Home~ ><",
    "Puppies - Urgent Adoption",
    "Puppies (2)",
    "Puppies (Stray)",
    "Puppies 3 Male M.B Rottweiler",
    "Puppies For Adoption :)",
    "Puppies For Foster / Adoption",
    "Puppies For Urgent Adoption",
    "Puppies Looking For Good Homes",
    "Puppies Looking For Homes",
    "Puppies Looking For Sweet Home",
    "Puppies Mixed Breed 3 Males",
    "Puppy - Female",
    "Puppy - Male",
    "Puppy (3)",
    "Puppy (5)",
    "Puppy (6)",
    "Puppy 03",
    "Puppy A",
    "Puppy A And Puppy B",
    "Puppy B1",
    "Puppy For Adoption!!",
    "Puppy For Adoption..URGENT!!",
    "Puppy Looking For Good Home",
    "Puppy Looking For Home",
    "Puppy Needs A Home",
    "PUPPY R1",
    "PUPPY R2",
    "PUPPY R3",
    "PUPPY R4",
    "PUPPY R5",
    "Puppy Urgent Adopt !!!!",
    "Puppy X And Z",
    "Puppy!!",
    "Puppy.",
    "Pups",
    "Pups A To F",
    "Pups For Adoption",
    "PUPS FOR ADOPTION!",
    "Q1",
    "Q2",
    "Q3",
    "Q4",
    "Q5",
    "Q6",
    "Q7",
    "R10",
    "R11",
    "R12",
    "R2",
    "R3",
    "R5",
    "R6",
    "R7",
    "R9",
    "Rescue Kittens For Adoption",
    "Rescued Cats And Kitten",
    "Rescued Kittens And Cats",
    "Rescued Puppies",
    "Rescued Puppy",
    "Rescued Pups For Adoption",
    "Rescued Pups Or Dogs",
    "Rottweiler Puppy",
    "S",
    "S&P",
    "S1",
    "S2",
    "S3",
    "S4",
    "S618",
    "Shepherd",
    "Siamese Kitten",
    "Siamese Kitten Boy",
    "Siblings For 7 Pups",
    "Siblings For Adoption",
    "Six Pups",
    "Spotted White Puppy",
    "Stray Cat Needs Home",
    "Stray Cute Dogs",
    "Stray Kitten",
    "Stray Kittens",
    "Stray Mother And Kittens",
    "Stray Puppy Female",
    "Sweet Cat",
    "T1",
    "T2",
    "T3",
    "Tabby Kitten 2",
    "Terrier",
    "Terrier Mix Puppy",
    "The  5 Pups",
    "The 4 Kitties!",
    "The 5 Little Puppies",
    "The 6 Kitties",
    "The Four Kittens",
    "The Kittens",
    "The PUPPIES",
    "Three Cute Kitties",
    "Three Little Kittens - Tabby",
    "Three Little Kittens - White Female",
    "Three Little Kittens - White Male",
    "Three Siblings",
    "Two Kittens Mix Color Orange White",
    "Two Little Kittens",
    "Two Small Puppies",
    "Two Stray Puppies",
    "Two White Dogs",
    "Urgent ! For Adoption",
    "Urgent : Cats For FREE Adoption",
    "Urgent :3 Cute Kittens For Adoption",
    "Urgent Adoption - Kittens",
    "Urgent Adoption - Male Kitten",
    "Urgent! For Adoption",
    "URGENT!!Cute Kittens For Adoption!!",
    "URGENT!Female Cat With Her Three Kittens",
    "URGENT:Handsome Dog Needs A Home",
    "V6",
    "W+",
    "W1",
    "W2",
    "W3",
    "W4",
    "W5",
    "W6",
    "W7",
    "W8",
    "White Female Kitten",
    "White Kittens",
    "White Kitties For Free Adoption.",
    "White Male Kitten",
    "White Mongrel Female Dog",
    "White Schnauzer For Adoption",
    "White Siamese Kitten",
    "White, Male Kitten For Adoption",
    "Y0",
    "Y1",
    "Y2",
    "Y3",
    "Y4",
    "Y5",
    "Y6",
    "Y7",
    "Young Kittens",
    "Z",
    "Z1",
    "Z2",
    "Z3",
    "Z4"
]

    maturity_size_mode = int(df_train.loc[df_train['MaturitySize'] != 0, 'MaturitySize'].mode()[0])
    fur_length_mode = int(df_train.loc[df_train['FurLength'] != 0, 'FurLength'].mode()[0])

    for df in [df_train, df_test]:

        # has_name: 1 si el nombre no es NaN ni string vacío ni uno de los nombres genéricos
        df['has_name'] = (df['Name'].notna() & (df['Name'].str.strip() != '') & ~(df['Name'].str.strip().isin(generic_names)) ).astype(int)

        # Description: fill NaN con string vacío para funciones de texto
        df['Description'] = df['Description'].fillna('')

        df['MaturitySize'] = df['MaturitySize'].replace(0, maturity_size_mode)
        df['FurLength']    = df['FurLength'].replace(0, fur_length_mode)

    return df_train, df_test

### 5.2 Age Features

**Justificación EDA:** Age tiene outliers extremos (máx 255 meses ≈ 21 años). El percentil 99 de train sirve como clip para no distorsionar las transformaciones. Animales muy jóvenes (< 4 meses) suelen adoptarse más rápido → indicador `is_young`. La binificación convierte la relación no lineal edad-adopción en una señal más limpia para los árboles.

In [5]:
def add_age_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Clip de outliers, binificación y flag de animal joven.

    Parámetros
    ----------
    df_train : pd.DataFrame
    df_test : pd.DataFrame
   
    Dfs con Nuevas columnas: age_clipped, age_group (int 0-5), is_young (0/1).
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    age_clip_value = float(df_train['Age'].quantile(AGE_CLIP_PERCENTILE / 100))

    for df in [df_train, df_test]:
        df['age_clipped'] = df['Age'].clip(upper=age_clip_value)
        # include_lowest=True asegura que Age=0 caiga en el primer bin.
        # fillna defensivo por si quedan NaN (valores fuera de rango o nulos).
        df['age_group']   = pd.cut(
                df['age_clipped'], bins=AGE_BINS, labels=AGE_LABELS,
                right=True, include_lowest=True,
            ).astype('Int64').fillna(0).astype(int)
        df['is_young']    = (df['age_clipped'] <= AGE_YOUNG_THRESHOLD).astype(int)

    return df_train, df_test


### 5.3 Fee Features

**Justificación EDA:** Fee tiene mediana = 0 (la mayoría de mascotas son gratuitas) y rango 0-3000 MYR con fuerte sesgo. Un indicador binario captura la barrera de costo, `log1p` comprime la escala y los cuantiles crean categorías ordinales de precio.

In [6]:
def add_fee_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Indicador binario, log-transform y quantile bins de Fee.

    Parámetros
    ----------
    df_train : pd.DataFrame
    df_test : pd.DataFrame

    Nuevas columnas: has_fee, fee_log1p.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    for df in [df_train, df_test]:
        df['has_fee']    = (df['Fee'] > 0).astype(int)
        df['fee_log1p']  = np.log1p(df['Fee'])

    return df_train, df_test

### 5.4 Photo & Video Features

**Justificación EDA:** PhotoAmt varía de 0 a 30 y es uno de los features más importantes en múltiples soluciones top ("photos = vetrina de la mascota"). VideoAmt tiene distribución casi completamente en cero → solo `has_video` binario aporta señal real. `photos_per_animal` normaliza por la cantidad de mascotas en el listado, capturando si múltiples animales comparten pocas fotos.

In [7]:
def add_photo_video_features(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features derivadas de PhotoAmt y VideoAmt.

    Nuevas columnas: has_photo, has_video, total_media,
                     photo_video_ratio, photos_per_animal.
    Sin riesgo de leakage — aritmética pura.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    for df in [df_train, df_test]:
        df['has_photo']         = (df['PhotoAmt'] > 0).astype(int)
        df['has_video']         = (df['VideoAmt'] > 0).astype(int)
        df['total_media']       = df['PhotoAmt'] + df['VideoAmt']
        df['photo_video_ratio'] = df['PhotoAmt'] / (df['VideoAmt'] + 1)
        df['photos_per_animal'] = df['PhotoAmt'] / df['Quantity'].clip(lower=1)

    return df_train, df_test

### 5.5 Health Features

**Justificación EDA:** Vaccinated, Dewormed y Sterilized son ternarias (1=Sí, 2=No, 3=Unknown). El EDA muestra que el cuidado veterinario está correlacionado con adopción más rápida. Un score compuesto (0-3) captura el nivel total de cuidado, mientras que los indicadores binarios individuales permiten que el modelo aprenda efectos parciales.

> **Importante:** esta función debe llamarse **antes** de `preprocess_basic` para leer los códigos ternarios originales.

In [8]:
def add_health_features(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Score de salud compuesto e indicadores binarios individuales.

    Lee los códigos ternarios originales (1=Sí, 2=No, 3=Unknown).
    Debe llamarse ANTES de preprocess_basic.

    Nuevas columnas:
      health_score      — cantidad de columnas con valor 1 (confirmado Sí). Rango 0-3.
      vaccinated_bin    — 1 si Vaccinated == 1, else 0.
      dewormed_bin      — 1 si Dewormed == 1, else 0.
      sterilized_bin    — 1 si Sterilized == 1, else 0.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    for df in [df_train, df_test]:

        for col in HEALTH_VARS:
            df[f'{col.lower()}_bin'] = (df[col] == TERNARY_YES).astype(int)

        df['health_score'] = (
            df['vaccinated_bin'] + df['dewormed_bin'] + df['sterilized_bin']
        )

        """: Un adoptante busca minimizar la incertidumbre. Una mascota que tiene todas sus vacunas, está desparasitada y esterilizada representa un menor costo y riesgo inicial. Creamos un índice de "Cuidado Completo" que resuma estas tres variables en un solo valor numérico."""
        # Otro enfoque: penalizar la falta de información (Unknown) con -1 en vez de 0
        # Transformamos a: 1 (Yes), 0 (No), -1 (Not Sure) para penalizar la falta de información
        health_mapping = {1: 1, 2: 0, 3: -1}
        
        temp_vac = df['Vaccinated'].map(health_mapping)
        temp_dew = df['Dewormed'].map(health_mapping)
        temp_ste = df['Sterilized'].map(health_mapping)
        
        # 1. Score de Salud Total
        df['health_score_penalized'] = temp_vac + temp_dew + temp_ste
        
        #Flag de "Cuidado Total" (Perfect Health Record)
        df['fully_covered'] = ((df['Vaccinated'] == 1) & 
                            (df['Dewormed'] == 1) & 
                            (df['Sterilized'] == 1)).astype(int)
        
        """El valor 3 (Not Sure) es una señal de que el rescatista no conoce el pasado del animal. Esto ocurre mucho en animales rescatados de la calle recientemente. Esta "incertidumbre" suele correlacionar con una adopción más lenta."""
        # Contamos cuántas veces aparece el valor '3' (Not Sure) en las 3 categorías principales
        health_cols = ['Vaccinated', 'Dewormed', 'Sterilized']
        df['health_uncertainty_count'] = (df[health_cols] == 3).sum(axis=1)
        
        # Flag: ¿Sabemos algo de su salud?
        df['has_health_records'] = (df['health_uncertainty_count'] < 3).astype(int)


        """Justificación competencia: No es lo mismo un perro adulto no esterilizado (posible descuido o perro de criadero) que un cachorro de 2 meses no esterilizado (es demasiado joven para la cirugía). Esta variable ajusta la "falta de esterilización" según la edad."""
        # Asumimos que antes de los 6 meses (Age < 6) es normal no estar esterilizado
        df['is_legit_unsterilized'] = ((df['Sterilized'] == 2) & (df['Age'] < 6)).astype(int)
        
        # "Red Flag": Adulto (+12 meses) sin vacunas ni esterilización
        df['neglected_adult_flag'] = ((df['Age'] >= 12) & 
                                    (df['Vaccinated'] == 2) & 
                                    (df['Sterilized'] == 2)).astype(int)

    return df_train, df_test

### 5.6.1 Breed Features

**Justificación EDA + competencia:** Breed1 y Breed2 tienen cardinalidad 307 → one-hot encoding impracticable. Frequency encoding comprime en un float informativo (breeds raras vs. populares). El **K-Fold target encoding** fue el feature individual más importante en la solución del 10° lugar (+0.007 QWK): breeds cuyas mascotas se adoptan más rápido tienen un encoding más bajo. Se implementa con K-Fold para evitar leakage.

In [9]:
def add_breed_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Genera variables relacionadas con la pureza de la raza y la 
    complejidad de la mezcla genética de la mascota.

    Se recomienda pre-procesar BreedLabels para identificar palabras clave:
    'Terrier', 'Retriever', 'Siamese', 'Persian', etc.
    
    Args:
        df_train (pd.DataFrame): DataFrame con Breed1 y Breed2.
        df_test (pd.DataFrame): DataFrame con Breed1 y Breed2.
        
    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: Tupla con DataFrames con flags de 'is_purebred', 'is_long_hair_breed' e 'is_mixed'.
    """

    df_train = df_train.copy()
    df_test = df_test.copy()

    for df in [df_train, df_test]:
        # IDs de 'Mixed Breed' según el archivo BreedLabels.csv
        # Perro: 307, Gato: 265
        df['is_mixed'] = (
            (df['Breed2'] != 0) | 
            (df['Breed1'] == 307) | 
            (df['Breed1'] == 265) |
            (df['Breed1'] == 0) # Casos sin raza especificada
        ).astype(int)
        
        # Flag complementario: Raza Pura única
        df['is_purebred'] = ((df['Breed2'] == 0) & (df['Breed1'] != 307) & (df['Breed1'] != 265)).astype(int)

        # Ejemplo: Identificar razas de pelo largo (Long Hair) vs Corto
        long_hair_keywords = ['Long Hair', 'Persian', 'Main Coon', 'Golden Retriever']
        
        # Mapear nombres de Breed1
        breed_names = df['Breed1'].map(breed_labels_df.set_index('BreedID')['BreedName'])
        
        df['is_long_hair_breed'] = breed_names.str.contains('|'.join(long_hair_keywords), na=False).astype(int)

        
    return df_train, df_test

### 5.6.2 Breeed Target Encoding with K-Fold & Smoothing
**Justificación competencia:** El Target Encoding estándar es extremadamente propenso al overfitting. Implementamos una versión que utiliza Smoothing, la cual pondera la media de la categoría con la media global basándose en la frecuencia de la categoría. Cuanto menos frecuente sea una raza, más se acercará su valor a la media global, reduciendo el ruido y el data leakage.

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

def apply_safe_target_encoding(train_df, test_df, col='Breed1', target='AdoptionSpeed',
                                group_col='RescuerID', folds=5, smoothing=10):
    """
    Target Encoding con CV grupal y suavizado Bayesiano.

    Cambios respecto a la versión KFold simple:
    - Usa StratifiedGroupKFold con `group_col` (default RescuerID).
      Esto evita el leakage indirecto: pets del mismo rescuer NO se separan
      entre fold IF y fold OOF, así la media de Breed1 que se aprende para
      el OOF no está "contaminada" por los hermanos rescuer del mismo pet.
    - El nombre de la columna nueva ahora se deriva de `col` en lugar de
      estar hardcodeado a 'Breed1_enc' (bug previo).

    Args:
        train_df: dataset de entrenamiento (debe contener `target` y `group_col`).
        test_df: dataset de prueba.
        col: columna categórica a codificar (ej. 'Breed1').
        target: variable objetivo (ej. 'AdoptionSpeed').
        group_col: columna por la cual agrupar el split (ej. 'RescuerID').
        folds: cantidad de folds.
        smoothing: peso del suavizado (alpha).

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: train y test con la nueva columna.
    """
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()

    # Media global del target (solo en train)
    global_mean = train_df[target].mean()

    new_col_name = f'{col}_enc'
    train_encoded[new_col_name] = 0.0

    # ── PROCESO PARA TRAIN (Out-of-Fold) con grupos ───────────────────────
    sgkf = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=42)
    y_train = train_df[target].values
    groups  = train_df[group_col].values

    for train_idx, val_idx in sgkf.split(train_df, y_train, groups=groups):
        df_train_fold = train_df.iloc[train_idx]
        df_val_fold   = train_df.iloc[val_idx]

        agg = df_train_fold.groupby(col)[target].agg(['mean', 'count'])
        smooth_map = (
            (agg['count'] * agg['mean'] + smoothing * global_mean)
            / (agg['count'] + smoothing)
        )

        train_encoded.loc[train_encoded.index[val_idx], new_col_name] = (
            df_val_fold[col].map(smooth_map).fillna(global_mean)
        )

    # ── PROCESO PARA TEST: usa todo train para mapear ──────────────────────
    agg_full = train_df.groupby(col)[target].agg(['mean', 'count'])
    smooth_map_full = (
        (agg_full['count'] * agg_full['mean'] + smoothing * global_mean)
        / (agg_full['count'] + smoothing)
    )
    test_encoded[new_col_name] = (
        test_encoded[col].map(smooth_map_full).fillna(global_mean)
    )

    # Devuelve train_df y test_df con la nueva columna
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[new_col_name] = train_encoded[new_col_name].values
    test_df[new_col_name]  = test_encoded[new_col_name].values

    return train_df, test_df


### 5.7.1 Description Text Features (hand-crafted)

**Justificación EDA:** El EDA ya mostró que `desc_len` (longitud de la descripción) correlaciona con AdoptionSpeed. Aquí se amplía el set con estadísticas de texto más granulares: conteo de palabras, longitud promedio de palabra, signos de puntuación emocional, uso de mayúsculas y dígitos. Estos features son útiles sin necesidad de modelos NLP pesados y complementan el análisis de sentimiento.

In [11]:
def add_text_features(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features estadísticos de la columna Description.

    Espera que Description ya haya sido rellenada con '' para los NaN
    (hecho por preprocess_basic).

    Nuevas columnas:
      desc_len               — longitud en caracteres.
      desc_word_count        — cantidad de palabras.
      desc_avg_word_len      — longitud media de las palabras.
      desc_sentence_count    — oraciones aproximadas (separadas por . ! ?).
      desc_exclamation_count — cantidad de '!'.
      desc_question_count    — cantidad de '?'.
      desc_upper_ratio       — fracción de letras en mayúscula.
      desc_digit_count       — cantidad de dígitos.
    """

    generic_descriptions = [
    "For Adoption",
    "Dog 4 Adoption",
    "Cat for adoption",
    "Friendly",
    "Dog for adoption",
    "Please feel free to contact us : Stuart",
    "Kittens for adoption",
    "",
    "Puppy for adoption",
    "Kitten for adoption",
    "Puppies for adoption",
    "'-",
    "LOOKING FOR A FOREVER HOME FOR HER",
    "Cute",
    "active",
    "Please feel free to contact us : Stuart Stewie",
    "Please give me a home",
    "Friendly and cute",
    "healthy",
    "Playful",
    "Please adopt & save!",
    "Please contact : Stuart Stewie",
    "PLS CALL for this puppy",
    "Super friendly",
    ".",
    ":)",
    "Adorable",
    "Cats for adoption",
    "Cute puppy for adoption",
    "free",
    "Free adoption",
    "happy",
    "Hyper active",
    "Kindly contact us if interested in adopting this baby.",
    "Looking for a good home",
    "Open for adoption!!!",
    "Please call Jocelyn.",
    "Please call Melissa",
    "Please contact if you are interested.",
    "Please give her a home.",
    "Please give me one home",
    "puppies for adopt",
    "very active",
    "very cute",
    "very loving kitten",
    "very playful",
    "Whatsapps",
    "WhatsApps for more info",
    "'---",
    "*FREE ADOPTION* Looking for loving home~!!",
    "..",
    "。。。",
    "~pls call",
    "0",
    "2 PUPPIES FOR ADOPTION...",
    "2 semi stray kittens for adoption",
    "4 cute kittens for adoption.",
    "8 puppies for adoption, urgent...",
    "A lovely cat.",
    "Active and adorable",
    "active and adorable kitten",
    "active and adorable puppy. Quiet.",
    "active and adorable.",
    "Active and loving",
    "Active and obedient.",
    "Active and playful",
    "active cat",
    "Active dog",
    "Active, cute and smart",
    "active, friendly, lovable",
    "Active, loving and playful",
    "Active, playful and healthy",
    "active, playful, cute",
    "Adopt me pls",
    "Adorable and friendly :)",
    "Adorable and smart",
    "Adorable cat...",
    "adorable, playful and lovely puppy....",
    "adorable.",
    "adpoted",
    "An adorable active kitten. Super friendly.",
    "An playful and active dog",
    "Anyone interested ? Please do contact me =)",
    "Anyone interested ? Please do contact me.",
    "Anyone interested. Please do contact me :))",
    "asdf asd",
    "awaiting adoption kitten.",
    "beautiful girl ^^",
    "Beautiful, loving and friendly cat for adoption.",
    "Booked",
    "call / whatssApp",
    "Call or SMS if u interested. Tq",
    "Call or SMS. Tq",
    "call this number if ur interesting",
    "Calm and nice.",
    "Cat for adoption for Free",
    "clever kitten for adoption.",
    "Condition: healthy If interested, please WhatsApp me directly:",
    "Contact me for any enquiry about the puppy...",
    "Contact Ms Ooi for Snowy adoption at!",
    "cute & adorable",
    "Cute & Lovely.....",
    "Cute & smart !",
    "Cute and active dogs..",
    "Cute and active.",
    "Cute and adorable",
    "Cute and adorable puppy looking for new home.",
    "cute and clever",
    "cute and friendly",
    "Cute and playful",
    "Cute and playful.",
    "Cute and small",
    "cute beautiful puppies for adoption",
    "cute call:",
    "Cute cat for adoption",
    "Cute kitten",
    "Cute kittens. Anyone interested ?",
    "Cute little girl looking for new home",
    "Cute puppies",
    "cute puppy",
    "Cute, active and playful.",
    "cute, active, adorable",
    "Cute, Adorable, Healthy, Obedient",
    "Cute, Adorable, Obedient",
    "Cute, good cat, playful and active.",
    "cuteeeee",
    "DaH",
    "for adoption contact me at meowpinkly(at).com",
    "For Adoption please contact",
    "For Adoption please contact me",
    "for adoption puppy",
    "for adoption/sale",
    "for inquiries please contact",
    "For more info contact Alexis",
    "Four kittens for adoption Contact",
    "Free for adoption",
    "Friendly & Playful",
    "Friendly and active",
    "Friendly and adorable...",
    "Friendly cat!!",
    "Good dog",
    "Have anything just call me to ask . Tq",
    "He is so active. Please contact",
    "He needs a home. Contact me",
    "Healthy & active dog for adoption",
    "healthy & happy.",
    "healthy and active.",
    "Healthy and active..cute",
    "healthy and adorable",
    "Healthy and lovely cat",
    "Healthy and need a new home.",
    "Healthy and playful",
    "Healthy and smart.",
    "Healthy cat.",
    "'-Healthy dog",
    "Healthy playful young kitten",
    "Healthy, active and cute kittens for adoption.",
    "Healthy, active puppy for adoption",
    "Healthy,, Active,, Loving,, Adorable,, :)",
    "Healthy..",
    "hello... anyone interested do contact me further",
    "Hi all Darling is available for adoption",
    "I have 3 playful kittens for adoptio .",
    "If interested in adopting please contact Jocelyn at",
    "if interested please call - Rani",
    "If interested, pls call for details.",
    "indulgent",
    "inform me if interested",
    "Interested to adopt pls call Amy",
    "jambu",
    "just call me...",
    "Kindly call Joseph Chia at for more information.",
    "Kindly contact us if interested in adopting this baby boy.",
    "Kindly contact us if interested in adopting this little baby",
    "Kindly email me for more info",
    "Kitten For Adoption 🐈",
    "kitten need new home",
    "Kittens for adoption. Very playful. See pictures.",
    "Kittens need new home",
    "little kitten for adoption ～",
    "LOOKING FOR A FOREVER HOME FOR HER....",
    "LOOKING FOR A FOREVER HOME FOR HIM",
    "Looking for adopter.",
    "Looking for good home...",
    "Looking for urgent home",
    "Looking forever home",
    "Lovable playful healthy kitten",
    "Loving",
    "Loving Dog",
    "Loving, adorable and playful",
    "Loyal",
    "Manja",
    "Manja and cute, feel free to contact - Whatsapp",
    "Mongral puppys for adoption whatsapp me.",
    "More details please contact Jacklyn",
    "more info text me .",
    "Mum and four kittens. Adopted from IntanShahar info.",
    "nice looking cat",
    "NIL",
    "",
    "Obedient",
    "ooooo",
    "open for adoption!!",
    "Open for adoption.",
    "pets for free adoption",
    "Playful and active kitten.",
    "Playful and lovely",
    "playful and lovely cat :)",
    "Playful kitten.",
    "playful, active and healthy",
    "Playful, active, cute and good boy",
    "Playfull....",
    "Please call .",
    "Please call Fan for more of the pups",
    "please call for adoption.",
    "please call Jocelyn",
    "Please call Joseph Chia at for further information.",
    "Please call or sms after office hour...",
    "Please call or SMS if interested :",
    "Please call or sms if you intrested. Tq",
    "Please call or whatsapp Jocelyn at to adopt.",
    "Please call to adopt/view Luki.",
    "Please contact Daniel",
    "Please contact Jacqueline about us at.",
    "Please contact Kelly for more details.",
    "Please contact Mr Low if you are interested.",
    "Please contact Nooreen at )",
    "Please e-mail me for adoption.",
    "Please SMS.",
    "pls call me if interested",
    "PLS CALL OR TEXT KAREN",
    "Pls Call Or Text Karen Eng",
    "PLS GIVE HIM A HOME.",
    "Puppies available for adoption",
    "puppy",
    "Puppy looking for new home",
    "ready for adoption !!",
    "Shy",
    "small adorable kitten.",
    "Smart and obedient. Very playful",
    "Smart Puppy",
    "Smart, adorable and lovely.",
    "Smart, healthy and playful dog.",
    "Super adorable puppies up for adoption",
    "Sweet & active",
    "Sweet and adorable, plz whatsapp for more info.",
    "Sweet girl",
    "Two twins kitten for adoption",
    "Urgent need new adopters",
    "URGENT!",
    "VERY ACTIVE AND FRIENDLY",
    "very active and playful little kittens",
    "Very active, friendly and clever.",
    "Very active, healthy & playful",
    "very cute!",
    "very cute,healthy....",
    "very friendly and active",
    "very friendly and loving",
    "Very friendly and playful puppy...",
    "very friendly dog~~",
    "Very friendly,playful and cute cat.",
    "very healthy beautiful puppies for adoption",
    "very lovely and smart cat",
    "very loving and quiet dog.",
    "Very loving cat",
    "very obedient and cute girl :)",
    "Very playful and adorable.",
    "Very playful, cute and adorable",
    "Very small puppies",
    "VERY VERY FRIENDLY",
    "Whatapp me if you interested in meow",
    "xx",
    "Young and active"
    ]

    df_train = df_train.copy()
    df_test = df_test.copy()

    def avg_word_len(s: str) -> float:
        ws = s.split()
        return float(np.mean([len(w) for w in ws])) if ws else 0.0

    def upper_ratio(s: str) -> float:
        alpha = [c for c in s if c.isalpha()]
        if not alpha:
            return 0.0
        return sum(1 for c in alpha if c.isupper()) / len(alpha)


    for df in [df_train, df_test]:
        text = df['Description'].fillna('')

        df['desc_len']    = text.str.len()

        words = text.str.split()
        df['desc_word_count'] = words.str.len().fillna(0).astype(int)

    
        df['desc_avg_word_len']      = text.apply(avg_word_len)
        df['desc_sentence_count']    = text.str.count(r'[.!?]')
        df['desc_exclamation_count'] = text.str.count(r'!')
        df['desc_question_count']    = text.str.count(r'\?')

        # Personalized Description: 1 si Description tiene un texto diferente al generico
        df['has_personalized_description'] = (text.str.strip().isin(generic_descriptions) == False).astype(int)


    
        df['desc_upper_ratio'] = text.apply(upper_ratio)
        df['desc_digit_count'] = text.str.count(r'\d')

    return df_train, df_test

### 5.7.2 Description Text-Based Urgency Detection
**Justificación competencia:** El lenguaje utilizado por el rescatista puede transmitir una carga emocional de necesidad inmediata. Las palabras que denotan crisis o rescate (como urgent o abandoned) suelen acelerar la decisión del adoptante por un factor de empatía y escasez. Esta feature binaria captura esa "llamada a la acción" que los modelos de embeddings genéricos a veces diluyen.

In [12]:
def extract_urgency_features(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Identifica indicadores de urgencia y situaciones de rescate crítico 
    dentro del texto de la descripción.
    
    Args:
        df_train (pd.DataFrame): DataFrame de entrenamiento con la columna de texto.
        df_test (pd.DataFrame): DataFrame de prueba con la columna de texto.
        
    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: DataFrames con las nuevas variables 'is_urgent' y 'urgency_score'.
    """
    # 1. Definición de palabras clave por categorías
    urgency_keywords = [
        'urgent', 'immediate', 'emergency', 'last chance', 
        'deadline', 'plead', 'saving', 'help'
    ]
    rescue_keywords = [
        'rescue', 'abandoned', 'stray', 'found', 'save', 
        'street', 'dumped', 'poor'
    ]
    
    # Combinamos todas las palabras para una búsqueda rápida
    all_keywords = urgency_keywords + rescue_keywords
    
    for df in [df_train, df_test]:
        # Pre-procesamiento: minúsculas y limpieza básica para evitar falsos positivos
        descriptions = df['Description'].fillna("").str.lower()
    
        # 2. Variable Binaria: ¿Contiene al menos una palabra de urgencia?
        # Usamos regex con \b para asegurar que busque palabras completas
        pattern = r'\b(' + '|'.join(all_keywords) + r')\b'
        df['is_urgent'] = descriptions.str.contains(pattern, regex=True).astype(int)
    
        # 3. Urgency Score: Cantidad de palabras de urgencia encontradas
        # Esto mide la intensidad del mensaje
        df['urgency_score'] = descriptions.apply(lambda x: sum(1 for word in all_keywords if word in x))
    
        # 4. Interaction: Urgencia + Edad
        # Un cachorro urgente suele tener prioridad máxima sobre un adulto urgente
        df['urgent_puppy_flag'] = ((df['is_urgent'] == 1) & (df['Age'] <= 6)).astype(int)
    
    return df_train, df_test

### 5.7.3 Description Text Embeddings & Dimensionality Reduction (SBERT + PCA)
**Justificación competencia:** Las descripciones en PetFinder contienen información crítica (personalidad, salud, historia) que las variables categóricas no capturan. Los embeddings de transformadores (Sentence-BERT) permiten mapear estas descripciones a un espacio vectorial semántico, donde mascotas con historias similares quedan representadas por vectores cercanos. Aplicamos PCA para reducir la dimensionalidad de 768 a 16-32 componentes; esto previene la "maldición de la dimensionalidad" y permite que los modelos de Gradient Boosting integren el contexto textual sin sobreajustar.

**Nota de performance:** La inferencia con modelos Transformer es intensiva en cómputo. Se recomienda el uso de GPU para la generación de embeddings (~3-5 minutos para el dataset completo). El resultado se almacena en el DataFrame final para evitar re-procesamiento en cada iteración del entrenamiento.

In [13]:
from functools import lru_cache
from sklearn.decomposition import PCA


@lru_cache(maxsize=1)
def _get_sentence_model(model_name: str) -> SentenceTransformer:
    return SentenceTransformer(model_name)  # GPU default


def apply_nlp_embeddings(
    df_train,
    df_test,
    n_components: int = 24,
    model_name: str = 'paraphrase-multilingual-MiniLM-L12-v2',
    batch_size: int = 64,
):
    """
    Genera embeddings multilingües y reduce su dimensionalidad mediante PCA,
    evitando data leakage.

    La lógica correcta es:
    - Generar embeddings para train y test.
    - Ajustar PCA únicamente sobre los embeddings de train.
    - Aplicar el PCA ya ajustado sobre test.

    Parámetros
    ----------
    df_train : pd.DataFrame
        Dataset de entrenamiento.

    df_test : pd.DataFrame
        Dataset de test / validación / holdout.

    n_components : int
        Cantidad de componentes principales a generar.

    model_name : str
        Nombre del modelo SentenceTransformer.

    batch_size : int
        Tamaño de batch para la inferencia del encoder.

    Retorna
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        df_train y df_test con columnas desc_pca_* agregadas.
    """

    # Copias para evitar modificar los DataFrames originales por referencia
    df_train = df_train.copy()
    df_test = df_test.copy()

    # Textos
    train_text_data = df_train['Description'].fillna("none").tolist()
    test_text_data = df_test['Description'].fillna("none").tolist()

    # Modelo cacheado: no se recarga si ya fue inicializado en esta sesión
    model = _get_sentence_model(model_name)

    print(f"Generando embeddings para train: {len(train_text_data)} registros...")
    train_embeddings = model.encode(
        train_text_data,
        show_progress_bar=True,
        batch_size=batch_size
    )

    print(f"Generando embeddings para test: {len(test_text_data)} registros...")
    test_embeddings = model.encode(
        test_text_data,
        show_progress_bar=True,
        batch_size=batch_size
    )

    # PCA ajustado SOLO con train para evitar leakage
    pca_model = PCA(n_components=n_components, random_state=42)

    train_pca_features = pca_model.fit_transform(train_embeddings)
    test_pca_features = pca_model.transform(test_embeddings)

    explained = pca_model.explained_variance_ratio_.sum()
    print(f"Varianza explicada en train con {n_components} componentes: {explained:.2%}")

    if explained < 0.70:
        print("Varianza < 70% — considera aumentar n_components")

    # Columnas PCA
    pca_cols = [f'desc_pca_{i}' for i in range(n_components)]

    df_train_pca = pd.DataFrame(
        train_pca_features,
        columns=pca_cols,
        index=df_train.index
    )

    df_test_pca = pd.DataFrame(
        test_pca_features,
        columns=pca_cols,
        index=df_test.index
    )

    # Concatenar columnas al dataset original
    df_train = pd.concat([df_train, df_train_pca], axis=1)
    df_test = pd.concat([df_test, df_test_pca], axis=1)

    return df_train, df_test

### 5.7.4 Description Sentiment Analysis (DistilBERT)

**Justificación competencia:** El análisis de sentimiento sobre la descripción es una feature NLP que aparece en múltiples soluciones top. Descripciones más positivas están asociadas con adopciones más rápidas. Usamos DistilBERT fine-tuneado en SST-2 (disponible en la librería `transformers` del environment).

**Nota de performance:** la inferencia sobre ~15K registros en CPU tarda aprox. 10-20 minutos. El pipeline se cachea para no reinicializar el modelo. Setear `load_sentiment_pipeline=False` en `fit_feature_mappings` para saltear este paso si no hay GPU disponible.

In [14]:
%%script False
# Se skipea porque solo funciona en ingles
def add_sentiment_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    model_name: str = 'distilbert-base-uncased-finetuned-sst-2-english',
    batch_size: int = 32,
    max_length: int = 512,
):
    """
    Análisis de sentimiento sobre una columna de texto usando Hugging Face transformers.

    Esta función recibe train y test juntos para procesarlos en una sola invocación.
    No hay entrenamiento ni ajuste sobre los datos, por lo que no hay leakage como en PCA.
    El modelo solo se usa para inferencia.

    Parámetros
    ----------
    df_train : pd.DataFrame
        Dataset de entrenamiento.

    df_test : pd.DataFrame
        Dataset de test / validación / holdout.

    model_name : str
        Modelo HuggingFace.
        Por defecto DistilBERT SST-2, binario POSITIVE/NEGATIVE.

    batch_size : int
        Tamaño de batch para inferencia.

    max_length : int
        Máximo de tokens. La descripción se trunca si excede.

    Nuevas columnas
    ---------------
    desc_sentiment_score : float
        Probabilidad normalizada de positividad.

    desc_sentiment_label : int
        1 = POSITIVE, 0 = NEGATIVE o NEUTRAL.

    Retorna
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        df_train y df_test con columnas de sentimiento agregadas.
    """

    from transformers import pipeline as hf_pipeline

    df_train = df_train.copy()
    df_test = df_test.copy()

    print(f'Cargando modelo {model_name}...')

    sentiment_pipeline = hf_pipeline(
        'sentiment-analysis',
        model=model_name,
        truncation=True,
        max_length=max_length,
    )

    def _apply_sentiment_to_df(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
        texts = df['Description'].fillna('').tolist()

        scores = []
        labels = []

        print(f'Calculando sentimiento para {dataset_name}: {len(texts)} registros...')

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

            # Separar textos vacíos para evitar inferencia innecesaria
            non_empty_texts = []
            non_empty_positions = []

            batch_results = [None] * len(batch)

            for j, text in enumerate(batch):
                if str(text).strip() == '':
                    batch_results[j] = {'label': 'NEUTRAL', 'score': 0.5}
                else:
                    non_empty_texts.append(text)
                    non_empty_positions.append(j)

            if non_empty_texts:
                model_results = sentiment_pipeline(non_empty_texts)

                for pos, result in zip(non_empty_positions, model_results):
                    batch_results[pos] = result

            for r in batch_results:
                lbl = r['label']
                sc = float(r['score'])

                # Normalizar: score siempre representa intensidad positiva
                if 'NEG' in lbl.upper():
                    sc = 1.0 - sc
                    labels.append(0)

                elif 'POS' in lbl.upper():
                    labels.append(1)

                else:
                    # NEUTRAL u otras etiquetas no binarias
                    labels.append(0)

                scores.append(sc)

        df['desc_sentiment_english_score'] = scores
        df['desc_sentiment_english_label'] = labels

        return df

    df_train = _apply_sentiment_to_df(df_train, dataset_name='train')
    df_test = _apply_sentiment_to_df(df_test, dataset_name='test')

    return df_train, df_test

Couldn't find program: 'False'


In [15]:
def add_sentiment_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    model_name: str = 'nlptown/bert-base-multilingual-uncased-sentiment',
    batch_size: int = 8,   # CPU: batch chico para no llenar RAM
    max_length: int = 256, # CPU: trunca para limitar RAM (descripciones promedio < 200 tokens)
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Análisis de sentimiento multilingüe sobre Description usando Hugging Face transformers.

    Esta versión usa por defecto un modelo multilingüe, más adecuado para PetFinder Malaysia,
    donde puede haber descripciones en inglés, malayo u otros idiomas.

    No hay entrenamiento ni ajuste sobre los datos, por lo que no hay leakage.
    El modelo solo se usa para inferencia.

    Nuevas columnas
    ---------------
    desc_sentiment_score : float
        Score normalizado de sentimiento positivo entre 0 y 1.

        Para modelos tipo estrellas:
          1 star  -> 0.00
          2 stars -> 0.25
          3 stars -> 0.50
          4 stars -> 0.75
          5 stars -> 1.00

        Para modelos POSITIVE/NEGATIVE:
          POSITIVE -> score original
          NEGATIVE -> 1 - score original

    desc_sentiment_label : int
        1 = sentimiento positivo, 0 = negativo / neutral.
    """

    from transformers import pipeline as hf_pipeline

    df_train = df_train.copy()
    df_test = df_test.copy()

    print(f'Cargando modelo {model_name}...')

    sentiment_pipeline = hf_pipeline(
        'sentiment-analysis',
        model=model_name,
        tokenizer=model_name,
        truncation=True,
        max_length=max_length,
        device=-1,  # CPU forzado: bug driver RTX 5070 con BERT sostenido
    )

    def _normalize_sentiment_result(result: dict) -> tuple[float, int]:
        """
        Convierte la salida del pipeline a:
        - score positivo entre 0 y 1
        - label binario: 1 positivo, 0 no positivo
        """

        lbl = str(result['label']).upper()
        raw_score = float(result['score'])

        # Caso 1: modelos tipo nlptown, etiquetas como "1 star", "5 stars"
        star_match = re.search(r'([1-5])', lbl)

        if star_match:
            stars = int(star_match.group(1))

            # Normalización de 1-5 estrellas a escala 0-1
            positive_score = (stars - 1) / 4

            # Threshold configurable conceptualmente:
            # 4 o 5 estrellas se consideran positivo
            positive_label = int(positive_score >= 0.60)

            return positive_score, positive_label

        # Caso 2: modelos binarios tipo POSITIVE / NEGATIVE
        if 'NEG' in lbl:
            positive_score = 1.0 - raw_score
            positive_label = 0
            return positive_score, positive_label

        if 'POS' in lbl:
            positive_score = raw_score
            positive_label = 1
            return positive_score, positive_label

        # Caso 3: NEUTRAL u otra etiqueta desconocida
        return 0.5, 0

    def _apply_sentiment_to_df(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
        texts = df['Description'].fillna('').astype(str).tolist()

        scores = []
        labels = []

        print(f'Calculando sentimiento para {dataset_name}: {len(texts)} registros...')

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

            non_empty_texts = []
            non_empty_positions = []

            batch_results = [None] * len(batch)

            for j, text in enumerate(batch):
                if text.strip() == '':
                    batch_results[j] = {'label': 'NEUTRAL', 'score': 0.5}
                else:
                    non_empty_texts.append(text)
                    non_empty_positions.append(j)

            if non_empty_texts:
                model_results = sentiment_pipeline(non_empty_texts)

                for pos, result in zip(non_empty_positions, model_results):
                    batch_results[pos] = result

            for result in batch_results:
                positive_score, positive_label = _normalize_sentiment_result(result)

                scores.append(positive_score)
                labels.append(positive_label)

        df['desc_sentiment_multilingual_score'] = scores
        df['desc_sentiment_multilingual_label'] = labels

        return df

    df_train = _apply_sentiment_to_df(df_train, dataset_name='train')
    df_test = _apply_sentiment_to_df(df_test, dataset_name='test')

    return df_train, df_test

### 5.8 Geographic Features

**Justificación competencia:** El 10° lugar y otras soluciones top integraron datos censales de Malaysia por estado. La hipótesis es que estados más densamente poblados y urbanizados tienen más adoptantes potenciales buscando mascotas online → adopción más rápida. Los datos censales son fijos (no se derivan del target) → sin riesgo de leakage. `state_freq_enc` sí se aprende de train.

In [16]:
import numpy as np
import pandas as pd
from typing import Tuple


def add_geographic_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Features geográficas a nivel de estado malaysiano.

    Evita leakage calculando el frequency encoding únicamente sobre train
    y aplicándolo luego tanto a train como a test.

    Combina:
    - Frecuencia de listings por estado, aprendida solo de train.
    - Estadísticas censales hardcodeadas, que no generan leakage.

    Nuevas columnas:
      state_freq_enc    — frecuencia de listings en ese estado calculada en train.
      state_population  — población del estado.
      state_pop_density — densidad poblacional.
      state_gdp_pc      — GDP per cápita.
      state_urban_pct   — fracción de población urbana.

    Retorna
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        df_train y df_test con columnas geográficas agregadas.
    """

    df_train = df_train.copy()
    df_test = df_test.copy()

    # Frequency encoding aprendido SOLO de train
    state_freq_map = df_train['State'].value_counts().to_dict()

    # Estadísticas censales fijas
    pop_map = {
        k: v['population']
        for k, v in MALAYSIA_STATE_STATS.items()
    }

    density_map = {
        k: v['population'] / v['area_km2']
        for k, v in MALAYSIA_STATE_STATS.items()
    }

    gdp_map = {
        k: v['gdp_pc']
        for k, v in MALAYSIA_STATE_STATS.items()
    }

    urban_map = {
        k: v['urban_pct']
        for k, v in MALAYSIA_STATE_STATS.items()
    }

    # Fallbacks para estados no encontrados en MALAYSIA_STATE_STATS
    med_pop = float(np.median(list(pop_map.values())))
    med_density = float(np.median(list(density_map.values())))
    med_gdp = float(np.median(list(gdp_map.values())))
    med_urban = float(np.median(list(urban_map.values())))

    def _apply_geo_features(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Si un estado aparece en test pero no en train, se asigna 1.0
        # porque sería un estado "nuevo" para el modelo.
        df['state_freq_enc'] = df['State'].map(state_freq_map).fillna(1.0)

        df['state_population'] = df['State'].map(pop_map).fillna(med_pop)

        df['state_pop_density'] = df['State'].map(density_map).fillna(med_density)

        df['state_gdp_pc'] = df['State'].map(gdp_map).fillna(med_gdp)

        df['state_urban_pct'] = df['State'].map(urban_map).fillna(med_urban)

        return df

    df_train = _apply_geo_features(df_train)
    df_test = _apply_geo_features(df_test)

    return df_train, df_test

### 5.9.1 Rescuer Aggregate Features

**Justificación competencia:** Múltiples soluciones top utilizaron agregaciones a nivel de RescuerID como segunda capa del stacking. Un rescatista que lista muchas mascotas es probablemente una ONG o rescatista profesional con mayor experiencia y red de contactos → adopciones más rápidas. El fee promedio refleja la política de precios del rescatista.

In [17]:

def add_rescuer_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Agregaciones a nivel de RescuerID evitando data leakage.

    Las estadísticas por RescuerID se calculan únicamente sobre df_train
    y luego se aplican tanto a df_train como a df_test.

    Nuevas columnas:
      rescuer_listing_count   — cantidad de listings del rescatista en train.
      rescuer_avg_fee         — fee promedio del rescatista calculado en train.
      rescuer_breed_diversity — cantidad de breeds distintas que maneja en train.
      rescuer_type_ratio      — fracción de perros Type == 1 del rescatista en train.

    Retorna
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        df_train y df_test con columnas de RescuerID agregadas.
    """

    df_train = df_train.copy()
    df_test = df_test.copy()

    # Fallbacks calculados SOLO con train
    train_fee_median = df_train['Fee'].median()

    # Mapas aprendidos SOLO desde train
    rescuer_maps = {
        'count_map': df_train['RescuerID'].value_counts().to_dict(),

        'avg_fee_map': df_train
            .groupby('RescuerID')['Fee']
            .mean()
            .to_dict(),

        'breed_div_map': df_train
            .groupby('RescuerID')['Breed1']
            .nunique()
            .to_dict(),

        'type_ratio_map': df_train
            .groupby('RescuerID')['Type']
            .apply(lambda x: (x == 1).mean())
            .to_dict(),
    }

    def _apply_rescuer_features(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        df['rescuer_listing_count'] = (
            df['RescuerID']
            .map(rescuer_maps['count_map'])
            .fillna(1)
            .astype(float)
        )

        df['rescuer_avg_fee'] = (
            df['RescuerID']
            .map(rescuer_maps['avg_fee_map'])
            .fillna(train_fee_median)
            .astype(float)
        )

        df['rescuer_breed_diversity'] = (
            df['RescuerID']
            .map(rescuer_maps['breed_div_map'])
            .fillna(1)
            .astype(float)
        )

        df['rescuer_type_ratio'] = (
            df['RescuerID']
            .map(rescuer_maps['type_ratio_map'])
            .fillna(0.5)
            .astype(float)
        )

        return df

    df_train = _apply_rescuer_features(df_train)
    df_test = _apply_rescuer_features(df_test)

    return df_train, df_test

### 5.9.2 Advanced Rescuer Effort & Care Features
**Justificación competencia:** Más allá de cuántos animales tiene un rescatista, importa cómo los presenta y qué tan responsable es. Rescatistas que consistentemente esterilizan a sus animales o que escriben descripciones detalladas para todos sus perfiles demuestran un estándar de calidad que el adoptante percibe y valora, acelerando la adopción.

In [18]:
def add_advanced_rescuer_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Agregaciones avanzadas a nivel de RescuerID evitando data leakage.

    Todas las estadísticas por RescuerID se calculan únicamente sobre df_train
    y luego se aplican tanto a df_train como a df_test.

    Nuevas columnas:
      rescuer_sterilization_rate  — fracción de mascotas esterilizadas por rescatista.
      rescuer_avg_description_len — longitud promedio de descripciones por rescatista.
      rescuer_health_completeness — promedio de salud: Vaccinated + Dewormed + Sterilized.
      rescuer_photo_effort        — promedio de fotos por listing del rescatista.

    Retorna
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        df_train y df_test con columnas avanzadas de RescuerID agregadas.
    """

    df_train = df_train.copy()
    df_test = df_test.copy()

    def health_score(df: pd.DataFrame) -> pd.Series:
        """
        Score de salud por fila:
        - Vaccinated == 1 suma 1
        - Dewormed == 1 suma 1
        - Sterilized == 1 suma 1

        En PetFinder:
        1 = Yes
        2 = No
        3 = Not Sure
        """
        return (
            (df['Vaccinated'] == 1).astype(int)
            + (df['Dewormed'] == 1).astype(int)
            + (df['Sterilized'] == 1).astype(int)
        )

    # Auxiliares calculados SOLO sobre train
    train_desc_len = df_train['Description'].fillna("").str.len()
    train_health_score = health_score(df_train)

    # Fallbacks calculados SOLO con train
    global_sterilization_rate = float((df_train['Sterilized'] == 1).mean())
    global_avg_description_len = float(train_desc_len.mean())
    global_health_completeness = float(train_health_score.mean())
    global_photo_effort = float(df_train['PhotoAmt'].mean())

    # Para evitar problemas si alguna media queda NaN
    if pd.isna(global_sterilization_rate):
        global_sterilization_rate = 0.0

    if pd.isna(global_avg_description_len):
        global_avg_description_len = 0.0

    if pd.isna(global_health_completeness):
        global_health_completeness = 0.0

    if pd.isna(global_photo_effort):
        global_photo_effort = 1.0

    # Dataset temporal solo para calcular agregaciones en train
    train_tmp = df_train.copy()
    train_tmp['_desc_len'] = train_desc_len
    train_tmp['_health_score'] = train_health_score

    # Mapas aprendidos SOLO desde train
    rescuer_maps = {
        'sterilization_map': (
            train_tmp
            .groupby('RescuerID')['Sterilized']
            .apply(lambda x: (x == 1).mean())
            .to_dict()
        ),

        'desc_len_map': (
            train_tmp
            .groupby('RescuerID')['_desc_len']
            .mean()
            .to_dict()
        ),

        'health_map': (
            train_tmp
            .groupby('RescuerID')['_health_score']
            .mean()
            .to_dict()
        ),

        'photo_effort_map': (
            train_tmp
            .groupby('RescuerID')['PhotoAmt']
            .mean()
            .to_dict()
        ),
    }

    def _apply_advanced_rescuer_features(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        df['rescuer_sterilization_rate'] = (
            df['RescuerID']
            .map(rescuer_maps['sterilization_map'])
            .fillna(global_sterilization_rate)
            .astype(float)
        )

        df['rescuer_avg_description_len'] = (
            df['RescuerID']
            .map(rescuer_maps['desc_len_map'])
            .fillna(global_avg_description_len)
            .astype(float)
        )

        df['rescuer_health_completeness'] = (
            df['RescuerID']
            .map(rescuer_maps['health_map'])
            .fillna(global_health_completeness)
            .astype(float)
        )

        df['rescuer_photo_effort'] = (
            df['RescuerID']
            .map(rescuer_maps['photo_effort_map'])
            .fillna(global_photo_effort)
            .astype(float)
        )

        return df

    df_train = _apply_advanced_rescuer_features(df_train)
    df_test = _apply_advanced_rescuer_features(df_test)

    return df_train, df_test

### 5.10.1 Interaction Features

**Justificación EDA:** El EDA mostró que perros y gatos tienen patrones de adopción distintos en salud, edad y estado. Las interacciones multiplicativas permiten al modelo capturar señales conjuntas sin necesidad de árboles más profundos. `urban_x_fee` captura la dinámica mercado premium (zona urbana + fee alto) vs. adopción social (zona rural + gratis).

In [19]:
def add_interaction_features(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Features de interacción multiplicativas.

    Requiere que las siguientes columnas ya existan:
      Type, Health, age_group, PhotoAmt, fee_log1p, Quantity,
      health_score, state_urban_pct.

    Nuevas columnas:
      type_x_health    — Type * Health.
      type_x_age_group — Type * age_group.
      photo_x_fee_log  — PhotoAmt * fee_log1p.
      quantity_x_photo — Quantity * PhotoAmt.
      urban_x_fee      — state_urban_pct * fee_log1p.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    for df in [df_train, df_test]:
      df['type_x_health']    = df['Type'] * df['Health']
      df['photo_x_fee_log']  = df['PhotoAmt'] * df['fee_log1p']
      df['quantity_x_photo'] = df['Quantity'] * df['PhotoAmt']
      df['urban_x_fee']      = df['state_urban_pct'] * df['fee_log1p']
      df['type_x_age_group'] = df['Type'] * df['age_group']
      df['fee_x_health'] = df['fee_log1p'] * df['health_score']
      df['type_x_urban'] = df['Type'] * df['state_urban_pct']
      df['type_x_sterilization'] = df['Type'] * df['Sterilized']
    

    return df_train, df_test

### 5.10.2 Advanced Interaction Features
**Justificación competencia:** Las interacciones añadidas buscan penalizar o bonificar perfiles basándose en la coherencia. Por ejemplo, un animal con problemas de salud (health_severity) en un estado con baja urbanización puede tener dinámicas de adopción distintas a uno en Kuala Lumpur. Asimismo, cruzamos el esfuerzo visual con la "atractividad" de la raza.

In [20]:
def add_advanced_interactions(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Crea interacciones complejas para capturar la coherencia del perfil 
    y el valor percibido por el adoptante.

    Requiere que existan:
      Age, MaturitySize, FurLength, Health, PhotoAmt, 
      is_purebred, rescuer_listing_count, desc_len.

    Nuevas columnas:
      size_x_furlength     — Interacción de apariencia física.
      age_x_size           — Coherencia de crecimiento (especialmente en cachorros).
      purebred_x_health    — Tolerancia de salud según pedigree.
      rescuer_vol_x_photo  — Nivel de profesionalismo (Volumen * Esfuerzo Visual).
      desc_len_x_photo     — Densidad de información total del listing.
      health_x_size        — Impacto de la fragilidad médica según el tamaño.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()

    for df in [df_train, df_test]:
        # 1. Apariencia Física: Animales grandes de pelo largo vs corto 
        # (Influye en la percepción de mantenimiento/belleza)
        df['size_x_furlength'] = df['MaturitySize'] * df['FurLength']

        # 2. Coherencia de Crecimiento: 
        # Un MaturitySize grande (3) con Age baja (cachorro) vs Adulto.
        df['age_x_size'] = df['Age'] * df['MaturitySize']

        # 3. Valor Percibido vs Salud: 
        # ¿Es una raza pura con problemas médicos? (is_purebred debe ser binaria 0/1)
        df['purebred_x_health'] = df['is_purebred'] * df['Health']

        # 4. Profesionalismo del Rescatista: 
        # Cruza el volumen de listings con el esfuerzo en fotos del listing actual.
        df['rescuer_vol_x_photo'] = df['rescuer_listing_count'] * df['PhotoAmt']

        # 5. Densidad de Información: 
        # Interacción entre riqueza visual y textual.
        df['desc_len_x_photo'] = df['desc_len'] * df['PhotoAmt']

        # 6. Fragilidad Médica: 
        # Problemas de salud en animales muy pequeños suelen percibirse como más críticos.
        # Invertimos MaturitySize para que 'Small' tenga más peso si Health es alto.
        df['health_x_size_inv'] = df['Health'] * (4 - df['MaturitySize'])

        # 7. Esfuerzo de Rescate: 
        # ¿Es un rescatista de bajo volumen poniendo mucho esfuerzo en descripción?
        # Ayuda a identificar rescatistas individuales muy dedicados.
        df['low_vol_high_desc'] = (1 / (df['rescuer_listing_count'] + 1)) * df['desc_len']

    return df_train, df_test

### 5.11 Quantity Features

**Justificación EDA:** Quantity varía de 1 a 20. Un solo animal se adopta de forma diferente a un grupo. La transformación logarítmica comprime la cola derecha.

In [21]:
def add_quantity_features(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Indicador de animal único y log-transform de Quantity.

    Nuevas columnas: is_single, quantity_log.
    """
    df_train = df_train.copy()
    df_test = df_test.copy()
    
    for df in [df_train, df_test]:
        df['is_single']    = (df['Quantity'] == 1).astype(int)
        df['quantity_log'] = np.log1p(df['Quantity'])
        
    return df_train, df_test

### 5.12 Color Features

**Justificación EDA:** Color1 siempre tiene valor (≥1), Color2 y Color3 son 0 si no aplica. La cantidad de colores distintos es un proxy de variedad visual de la mascota.

In [22]:
def add_color_features(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Cantidad de colores distintos reportados.

    Nueva columna: num_colors (1–3).
    """
    df_train = df_train.copy()
    df_test = df_test.copy()
    
    for df in [df_train, df_test]:
        df['num_colors'] = (df[['Color1', 'Color2', 'Color3']] != 0).sum(axis=1)
    
    return df_train, df_test

---
## 6. Feature Orchestration


In [23]:
def build_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Pipeline completo de feature engineering.

    Aplica las 13 funciones de feature engineering en orden.
    Elimina columnas raw al final.

    Parámetros
    ----------
    df_train : pd.DataFrame
        DataFrame de entrenamiento crudo. NO debe contener la columna AdoptionSpeed.
    df_test : pd.DataFrame
        DataFrame de prueba crudo. NO debe contener la columna AdoptionSpeed.
    Retorna
    -------
    pd.DataFrame con todas las features engineeradas y columnas raw dropeadas.
    """


    df_train = df_train.copy()
    df_test = df_test.copy()


    print('Building features...')
    print('  Health features...')
    # 1. Health features PRIMERO (lee códigos ternarios originales)
    df_out_train, df_out_test = add_health_features(df_train, df_test)

    print('  Basic preprocessing...')
    # 2. Preprocessing básico
    df_out_train, df_out_test = preprocess_basic(df_out_train, df_out_test)

    print('  Age features...')
    # 3. Age features
    df_out_train, df_out_test = add_age_features(df_out_train, df_out_test)

    print('  Fee features...')

    # 4. Fee features
    df_out_train, df_out_test = add_fee_features(df_out_train, df_out_test)

    print('  Photo & Video features...')

    # 5. Photo & Video features
    df_out_train, df_out_test = add_photo_video_features(df_out_train, df_out_test)

    print('  Breed features...')

    print('     Breed features...')
    # 6. Breed features
    df_out_train, df_out_test = add_breed_features(df_out_train, df_out_test)

    print('     Advanced Breed features (Target Encoding)...')
    # 7. Target Encoding Breed features
    df_out_train, df_out_test = apply_safe_target_encoding(df_out_train, df_out_test)
    

    print('  Text features...')
    # 8. Text features completas, incluyendo análisis de urgencia y características básicas.    
    
    print('     Text features Basics...')
    df_out_train, df_out_test = add_text_features(df_out_train, df_out_test)

    print('     Text features Urgency...')
    df_out_train, df_out_test = extract_urgency_features(df_out_train, df_out_test)

    print('     Text features Embeddings...')
    df_out_train, df_out_test = apply_nlp_embeddings(df_out_train, df_out_test)

    print('     Text features Sentiment...')
    df_out_train, df_out_test = add_sentiment_features(df_out_train, df_out_test)

    print('  Geographic features...')
    # 9. Geographic features
    df_out_train, df_out_test = add_geographic_features(df_out_train, df_out_test)

    print('  Rescuer features...')  
    # 10. Rescuer features

    print('     Rescuer features Aggregations...')  
    df_out_train, df_out_test = add_rescuer_features(df_out_train, df_out_test)
    
    print('     Rescuer features AggAdvanced...')  
    df_out_train, df_out_test = add_advanced_rescuer_features(df_out_train, df_out_test)
    
    print('  Quantity features...')
    # 11. Quantity features
    df_out_train, df_out_test = add_quantity_features(df_out_train, df_out_test)
    
    print('  Color features...')
    # 12. Color features
    df_out_train, df_out_test = add_color_features(df_out_train, df_out_test)

    print('  Interaction features...')
    # 13. Interaction features
    
    print('     Interaction features Basics...')
    df_out_train, df_out_test = add_interaction_features(df_out_train, df_out_test)

    print('     Interaction features Advanced...')
    df_out_train, df_out_test = add_advanced_interactions(df_out_train, df_out_test)

    print('  Dropping raw columns...')
    # Drop columnas raw
    df_out_train.drop(columns=COLS_TO_DROP, errors='ignore', inplace=True)
    df_out_test.drop(columns=COLS_TO_DROP, errors='ignore', inplace=True)

    return df_out_train, df_out_test

---
## 6. Modelo Baseline LGBM (pre-Feature Engineering)

Antes de aplicar el pipeline de feature engineering construimos un baseline con
**LightGBM** sobre las columnas crudas, para tener una referencia objetiva contra
la cual medir cuánto aporta el FE en términos de QWK / F1 / Accuracy.

**Decisiones de diseño del baseline**

- Solo dropeamos columnas que no aportan al modelo (texto libre, identificadores).
- No tocamos la codificación de las variables categóricas: LightGBM puede manejar
  enteros directamente; las columnas son originalmente numéricas en el CSV.
- Entrenamos sobre `df_train` y evaluamos sobre `df_test` (mismo split que se usará
  para el modelo final). De esta forma la comparación es honesta.
- Uso parámetros mínimos / por defecto a propósito: el objetivo no es maximizar
  el baseline, sino tener un piso reproducible.


In [24]:
# ── Baseline: features y datos crudos ──────────────────────────────────────
# El target se binariza: 1 si pertenece a BINARY_TARGET_CLASS, 0 caso contrario.
BASELINE_DROP_COLS = ['PetID', 'Name', 'Description', 'RescuerID']

baseline_features = [
    c for c in df_train.columns
    if c not in BASELINE_DROP_COLS + [LABEL]
]

X_train_base = df_train[baseline_features].copy()
y_train_base = (df_train[LABEL] == BINARY_TARGET_CLASS).astype(int)

X_test_base  = df_test[baseline_features].copy()
y_test_base  = (df_test[LABEL] == BINARY_TARGET_CLASS).astype(int)

if USE_META_TRAIN:
    X_meta_base  = df_meta_train[baseline_features].copy()
    y_meta_base  = (df_meta_train[LABEL] == BINARY_TARGET_CLASS).astype(int)
else:
    X_meta_base = y_meta_base = None

print(f'Baseline binario — target {BINARY_LABEL}')
print(f'  Distribucion train: {y_train_base.value_counts(normalize=True).round(3).to_dict()}')
print(f'  Distribucion test:  {y_test_base.value_counts(normalize=True).round(3).to_dict()}')
print(f'  Train: {X_train_base.shape}')
if USE_META_TRAIN:
    print(f'  Meta:  {X_meta_base.shape}')
print(f'  Test:  {X_test_base.shape}')


Baseline binario — target is_class_0
  Distribucion train: {0: 0.973, 1: 0.027}
  Distribucion test:  {0: 0.973, 1: 0.027}
  Train: (11994, 19)
  Test:  (2999, 19)


In [25]:
# ── Baseline: entrenamiento LGBM binario con parámetros por defecto ────────
lgb_params_baseline = {
    'objective'   : 'binary',
    'verbosity'   : -1,
    'random_state': SEED,
}

lgb_train_baseline = lgb.Dataset(
    data=X_train_base,
    label=y_train_base,
    free_raw_data=False,
)

baseline_model = lgb.train(
    lgb_params_baseline,
    lgb_train_baseline,
    num_boost_round=100,
)

# Predicciones probabilidad de clase positiva (= clase 0 del problema original)
y_prob_test_baseline = baseline_model.predict(X_test_base)
y_pred_test_baseline = (y_prob_test_baseline >= 0.5).astype(int)

if USE_META_TRAIN:
    y_prob_meta_baseline = baseline_model.predict(X_meta_base)
    y_pred_meta_baseline = (y_prob_meta_baseline >= 0.5).astype(int)
else:
    y_prob_meta_baseline = y_pred_meta_baseline = None

print('Baseline binario entrenado.')


Baseline binario entrenado.


In [26]:
# ── Baseline: métricas binarias ────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score

auc_baseline_test  = roc_auc_score(y_test_base, y_prob_test_baseline)
ap_baseline_test   = average_precision_score(y_test_base, y_prob_test_baseline)
f1_baseline_test   = f1_score(y_test_base, y_pred_test_baseline)
acc_baseline_test  = accuracy_score(y_test_base, y_pred_test_baseline)
bacc_baseline_test = balanced_accuracy_score(y_test_base, y_pred_test_baseline)

if USE_META_TRAIN:
    auc_baseline_meta  = roc_auc_score(y_meta_base, y_prob_meta_baseline)
    ap_baseline_meta   = average_precision_score(y_meta_base, y_prob_meta_baseline)
    f1_baseline_meta   = f1_score(y_meta_base, y_pred_meta_baseline)
    acc_baseline_meta  = accuracy_score(y_meta_base, y_pred_meta_baseline)
    bacc_baseline_meta = balanced_accuracy_score(y_meta_base, y_pred_meta_baseline)
else:
    auc_baseline_meta = ap_baseline_meta = f1_baseline_meta = acc_baseline_meta = bacc_baseline_meta = None

print('=' * 55)
print('BASELINE BINARIO — TEST')
print(f'  AUC                : {auc_baseline_test:.4f}')
print(f'  Avg Precision (PR) : {ap_baseline_test:.4f}')
print(f'  F1 (clase 0)       : {f1_baseline_test:.4f}')
print(f'  Accuracy           : {acc_baseline_test:.4f}')
print(f'  Balanced Accuracy  : {bacc_baseline_test:.4f}')
if USE_META_TRAIN:
    print('-' * 55)
    print('BASELINE BINARIO — META_TRAIN')
    print(f'  AUC                : {auc_baseline_meta:.4f}')
    print(f'  Avg Precision (PR) : {ap_baseline_meta:.4f}')
    print(f'  F1 (clase 0)       : {f1_baseline_meta:.4f}')
    print(f'  Accuracy           : {acc_baseline_meta:.4f}')
    print(f'  Balanced Accuracy  : {bacc_baseline_meta:.4f}')
print('=' * 55)

plot_confusion_matrix(
    y_test_base, y_pred_test_baseline,
    title=f'Baseline Binario (TEST) — AUC={auc_baseline_test:.4f}',
).show()


BASELINE BINARIO — TEST
  AUC                : 0.6459
  Avg Precision (PR) : 0.1032
  F1 (clase 0)       : 0.0909
  Accuracy           : 0.9733
  Balanced Accuracy  : 0.5240


---
## 7. Aplicar Feature Engineering y Re-Split

`build_features(df_train, df_test)` espera dos dataframes y trata el segundo como
"test". El pipeline interno (especialmente target encoding y agregaciones por
RescuerID/State) se ajusta sobre `df_train` y se aplica a `df_test`. Por lo tanto
**no debemos pasar `df_meta_train` por separado** porque se trataría como un nuevo
"train" y leakearía estadísticas.

**Estrategia**

1. Concatenar `df_meta_train + df_test` como un único `df_test_unified`.
2. Capturar `PetID` y la cantidad de filas de cada parte para poder re-splitear
   sin ambigüedad después.
3. Llamar `build_features(df_train, df_test_unified)`.
4. Re-splitear el resultado preservando exactamente el mismo universo (mismos
   `PetID`s) que tenía cada subset antes del FE.
5. Validar con `assert` que los `PetID`s coinciden.


In [27]:
# ── Aplicar (o cargar) Feature Engineering ────────────────────────────────
# Captura de PetIDs originales para preservar y validar el universo de cada DF.
# Esto es CRÍTICO porque luego entran a un metamodelo (en modo 3-way) o
# directamente al modelo final (2-way), y el alineamiento fila-a-fila importa.
import os

petids_train_orig = df_train['PetID'].tolist()
petids_test_orig  = df_test['PetID'].tolist()
petids_meta_orig  = df_meta_train['PetID'].tolist() if USE_META_TRAIN else []

# RescuerID también se preserva — necesario para el `groups=` de
# StratifiedGroupKFold (evita leakage indirecto por rescuer).
rescuers_train_orig = df_train['RescuerID'].tolist()
rescuers_test_orig  = df_test['RescuerID'].tolist()
rescuers_meta_orig  = df_meta_train['RescuerID'].tolist() if USE_META_TRAIN else []

n_meta = len(df_meta_train) if USE_META_TRAIN else 0
n_test = len(df_test)

os.makedirs(FE_OUTPUT_DIR, exist_ok=True)
fe_paths = {k: os.path.join(FE_OUTPUT_DIR, v) for k, v in FE_FILES.items()}

if PROCESS_FE:
    print('PROCESS_FE = True  →  calculando feature engineering...')

    # ── Concatenar meta_train + test (en ese orden) ───────────────────────
    # build_features espera df_train (con AdoptionSpeed) y df_test (sin él).
    # Cuando hay meta_train, lo unimos con test para una única invocación.
    if USE_META_TRAIN:
        df_test_unified = pd.concat([df_meta_train, df_test], axis=0, ignore_index=True)
    else:
        df_test_unified = df_test.copy().reset_index(drop=True)

    y_train_raw   = df_train[LABEL].copy()
    y_unified_raw = df_test_unified[LABEL].copy()

    df_train_in   = df_train.copy()                          # con AdoptionSpeed
    df_unified_in = df_test_unified.drop(columns=[LABEL])    # sin AdoptionSpeed

    # ── Aplicar pipeline completo ──────────────────────────────────────────
    df_train_fe, df_unified_fe = build_features(df_train_in, df_unified_in)

    # ── Re-split por orden posicional ──────────────────────────────────────
    if USE_META_TRAIN:
        df_meta_train_fe = df_unified_fe.iloc[:n_meta].reset_index(drop=True).copy()
        df_test_fe       = df_unified_fe.iloc[n_meta:].reset_index(drop=True).copy()
    else:
        df_meta_train_fe = None
        df_test_fe       = df_unified_fe.reset_index(drop=True).copy()
    df_train_fe = df_train_fe.reset_index(drop=True).copy()

    # ── Re-asociar el target (estaba excluido durante el FE en unified) ────
    df_train_fe[LABEL] = y_train_raw.reset_index(drop=True)
    df_test_fe[LABEL]  = y_unified_raw.iloc[n_meta:].reset_index(drop=True)
    if USE_META_TRAIN:
        df_meta_train_fe[LABEL] = y_unified_raw.iloc[:n_meta].reset_index(drop=True)

    # ── Re-incorporar PetID y RescuerID (PetID lo dropea build_features,
    #    RescuerID se mantiene en el DF post-FE para usarlo como `groups`
    #    en StratifiedGroupKFold). Reasignamos por seguridad. ──────────────
    df_train_fe['PetID']     = petids_train_orig
    df_test_fe['PetID']      = petids_test_orig
    df_train_fe['RescuerID'] = rescuers_train_orig
    df_test_fe['RescuerID']  = rescuers_test_orig
    if USE_META_TRAIN:
        df_meta_train_fe['PetID']     = petids_meta_orig
        df_meta_train_fe['RescuerID'] = rescuers_meta_orig

    # ── Persistir a disco (parquet preserva dtypes mejor que csv) ─────────
    df_train_fe.to_parquet(fe_paths['train'], index=False)
    df_test_fe.to_parquet(fe_paths['test'],   index=False)
    print(f'  Guardado:  {fe_paths["train"]}')
    print(f'  Guardado:  {fe_paths["test"]}')
    if USE_META_TRAIN:
        df_meta_train_fe.to_parquet(fe_paths['meta_train'], index=False)
        print(f'  Guardado:  {fe_paths["meta_train"]}')

else:
    print('PROCESS_FE = False  →  cargando features pre-calculadas desde disco...')

    needed_keys = ['train', 'test'] + (['meta_train'] if USE_META_TRAIN else [])
    for k in needed_keys:
        path = fe_paths[k]
        if not os.path.exists(path):
            raise FileNotFoundError(
                f'No existe {path}. Corré con PROCESS_FE=True al menos una vez '
                'para generar los parquet, o ajustá FE_OUTPUT_DIR/FE_FILES.'
            )

    df_train_fe = pd.read_parquet(fe_paths['train'])
    df_test_fe  = pd.read_parquet(fe_paths['test'])
    print(f'  Cargado:  {fe_paths["train"]}')
    print(f'  Cargado:  {fe_paths["test"]}')

    if USE_META_TRAIN:
        df_meta_train_fe = pd.read_parquet(fe_paths['meta_train'])
        print(f'  Cargado:  {fe_paths["meta_train"]}')
    else:
        df_meta_train_fe = None

# ── Validaciones del universo (independiente de la rama tomada) ────────────
assert len(df_train_fe) == len(petids_train_orig), 'df_train: cantidad de filas no coincide'
assert len(df_test_fe)  == n_test,                 'df_test: cantidad de filas no coincide'
assert set(df_train_fe['PetID']) == set(petids_train_orig), 'df_train: PetIDs cambiaron'
assert set(df_test_fe['PetID'])  == set(petids_test_orig),  'df_test: PetIDs cambiaron'
assert 'RescuerID' in df_train_fe.columns, 'df_train_fe perdio RescuerID'
assert 'RescuerID' in df_test_fe.columns,  'df_test_fe perdio RescuerID'

if USE_META_TRAIN:
    assert len(df_meta_train_fe) == n_meta, 'df_meta_train: cantidad de filas no coincide'
    assert set(df_meta_train_fe['PetID']) == set(petids_meta_orig), 'df_meta_train: PetIDs cambiaron'
    assert 'RescuerID' in df_meta_train_fe.columns, 'df_meta_train_fe perdio RescuerID'

# Mismas columnas en train y test post-FE
diff = set(df_train_fe.columns).symmetric_difference(set(df_test_fe.columns))
assert not diff, f'Columnas distintas entre train y test post-FE: {diff}'

print('\nValidacion OK — universo preservado.')
print(f'\nShapes post-FE:')
print(f'  df_train_fe      : {df_train_fe.shape}')
if USE_META_TRAIN:
    print(f'  df_meta_train_fe : {df_meta_train_fe.shape}')
print(f'  df_test_fe       : {df_test_fe.shape}')
print(f'\nFeatures totales (excluyendo PetID, RescuerID y AdoptionSpeed): {df_train_fe.shape[1] - 3}')


PROCESS_FE = False  →  cargando features pre-calculadas desde disco...
  Cargado:  ../input/petfinder-adoption-prediction/train/train_fe.parquet
  Cargado:  ../input/petfinder-adoption-prediction/train/test_fe.parquet

Validacion OK — universo preservado.

Shapes post-FE:
  df_train_fe      : (11994, 112)
  df_test_fe       : (2999, 112)

Features totales (excluyendo PetID, RescuerID y AdoptionSpeed): 109


In [28]:
# ── Definición de matrices X / y para el modelo final ──────────────────────
features_fe = [c for c in df_train_fe.columns if c not in [LABEL, 'PetID', 'RescuerID']]

# RescuerID se preserva por separado para usarlo como `groups` en
# StratifiedGroupKFold (no es feature predictiva).
groups_train = df_train_fe['RescuerID'].values

X_train = df_train_fe[features_fe].copy()
y_train_orig = df_train_fe[LABEL].copy()              # multiclase 0..4 (referencia)
y_train      = (y_train_orig == BINARY_TARGET_CLASS).astype(int)

X_test  = df_test_fe[features_fe].copy()
y_test_orig = df_test_fe[LABEL].copy()
y_test      = (y_test_orig == BINARY_TARGET_CLASS).astype(int)

if USE_META_TRAIN:
    X_meta = df_meta_train_fe[features_fe].copy()
    y_meta_orig = df_meta_train_fe[LABEL].copy()
    y_meta      = (y_meta_orig == BINARY_TARGET_CLASS).astype(int)
else:
    X_meta = y_meta = y_meta_orig = None

print(f'\nTarget binarizado: y == {BINARY_TARGET_CLASS}')
print(f'  Train positivos: {y_train.sum()} / {len(y_train)} ({y_train.mean()*100:.1f}%)')
print(f'  Test  positivos: {y_test.sum()} / {len(y_test)} ({y_test.mean()*100:.1f}%)')

# Encoding defensivo de columnas object (p. ej. desc_sentiment_label si no está mapeado)
for col in X_train.select_dtypes(include='object').columns:
    print(f'Encoding columna object: {col}')
    mapping = {v: i for i, v in enumerate(sorted(X_train[col].dropna().unique()))}
    X_train[col] = X_train[col].map(mapping)
    X_test[col]  = X_test[col].map(mapping)
    if USE_META_TRAIN:
        X_meta[col] = X_meta[col].map(mapping)

# Features categóricas conocidas (filtradas a las efectivamente presentes)
CAT_FEATURES_CANDIDATES = [
    'Type', 'Gender', 'MaturitySize', 'FurLength', 'Health',
    'vaccinated_bin', 'dewormed_bin', 'sterilized_bin',
    'is_young', 'has_fee', 'has_name', 'has_photo', 'has_video',
    'is_mixed_breed', 'is_single', 'desc_sentiment_label',
    'fee_quantile_bin', 'num_colors',
]
CAT_FEATURES = [f for f in CAT_FEATURES_CANDIDATES if f in features_fe]

print(f'\nFeatures totales : {len(features_fe)}')
print(f'Categóricas      : {len(CAT_FEATURES)}')
print(f'Numéricas        : {len(features_fe) - len(CAT_FEATURES)}')



Target binarizado: y == 0
  Train positivos: 328 / 11994 (2.7%)
  Test  positivos: 82 / 2999 (2.7%)

Features totales : 109
Categóricas      : 15
Numéricas        : 94


---
## 8. Optimización de Hiperparámetros con Optuna + 5-Fold CV

Usamos la misma estructura que el notebook `05_LGBM_Pipeline`, con dos diferencias:

1. **Solo entrenamos sobre `df_train`** (no se usa `df_meta_train` ni `df_test`
   durante la optimización).
2. **Manejo de desbalanceo de clase 0**: la clase 0 (`AdoptionSpeed=0`) tiene
   muy pocos ejemplos. Calculamos `sample_weight` con
   `compute_sample_weight('balanced', y_train)` y lo pasamos al `lgb.Dataset`.
   Esto le da más peso relativo a las clases minoritarias sin alterar la
   distribución de los datos.

**Espacio de búsqueda** (especificado en el lab):

| Hiperparámetro    | Rango       | Tipo        |
|-------------------|-------------|-------------|
| `num_leaves`      | 16 – 256    | int         |
| `max_depth`       | 3 – 12      | int         |
| `learning_rate`   | 0.01 – 0.3  | float (log) |
| `lambda_l1`       | 1e-8 – 10   | float (log) |
| `lambda_l2`       | 1e-8 – 10   | float (log) |
| `feature_fraction`| 0.4 – 1.0   | float       |
| `bagging_fraction`| 0.4 – 1.0   | float       |
| `bagging_freq`    | 1 – 7       | int         |
| `min_child_samples`| 5 – 100    | int         |

La métrica objetivo es **QWK CV** (Quadratic Weighted Kappa promediado entre folds).


In [29]:
# ── Sample weights balanceados (manejo del desbalanceo en clase 0) ─────────
from sklearn.utils.class_weight import compute_sample_weight

sample_weight_train = compute_sample_weight(class_weight='balanced', y=y_train)

# Verificación: el peso por clase es inverso a la frecuencia
import collections
weights_per_class = {}
for cls in sorted(y_train.unique()):
    mask = (y_train == cls).values
    weights_per_class[int(cls)] = float(sample_weight_train[mask].mean())

print('Distribución original (counts):')
print(y_train.value_counts().sort_index().to_string())
print('\nPeso medio por clase (balanced):')
for cls, w in weights_per_class.items():
    print(f'  Clase {cls}: {w:.4f}')


Distribución original (counts):
AdoptionSpeed
0    11666
1      328

Peso medio por clase (balanced):
  Clase 0: 0.5141
  Clase 1: 18.2835


In [30]:
# ── Métrica: AUC nativo de LightGBM ─────────────────────────────────────────
# Para clasificacion binaria, LightGBM tiene 'auc' built-in. La pasamos como
# `metric` en lgb_params abajo.

# Helper opcional: F1 de la clase positiva (clase 0 del problema original).
def lgb_custom_metric_f1_class0(dy_pred, dy_true):
    """F1 score de la clase positiva para LightGBM binary."""
    from sklearn.metrics import f1_score as _f1
    y_true = dy_true.get_label()
    y_pred = (dy_pred >= 0.5).astype(int)
    return 'f1_class0', _f1(y_true, y_pred), True


In [31]:
# ── Función objetivo Optuna: 5-fold CV con early stopping (binary) ─────────
import os

# Aseguramos que existan los directorios de artefactos
for path in [PATH_TO_MODELS, PATH_TO_TEMP_FILES, PATH_TO_OPTUNA_ARTIFACTS]:
    os.makedirs(path, exist_ok=True)

artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)


def cv_es_lgb_objective(trial):
    """Función objetivo binaria con StratifiedGroupKFold para evitar leakage por rescuer.

    Optimiza AUC promedio sobre 5 folds.
    """
    lgb_params = {
        'objective'        : 'binary',
        'metric'           : 'auc',
        'verbosity'        : -1,
        'random_state'     : SEED,
        # ── Espacio de búsqueda ────────────────────────────────────────────
        'num_leaves'       : trial.suggest_int('num_leaves', 16, 256),
        'max_depth'        : trial.suggest_int('max_depth', 3, 12),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'lambda_l1'        : trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2'        : trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'feature_fraction' : trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction' : trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq'     : trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }

    cv_score = 0.0
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    for fold_idx, (if_index, oof_index) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
        w_if = sample_weight_train[if_index]

        lgb_if_ds = lgb.Dataset(
            data=X_train.iloc[if_index],
            label=y_train.iloc[if_index],
            weight=w_if,
            categorical_feature=CAT_FEATURES,
            free_raw_data=False,
        )
        lgb_oof_ds = lgb.Dataset(
            data=X_train.iloc[oof_index],
            label=y_train.iloc[oof_index],
            categorical_feature=CAT_FEATURES,
            free_raw_data=False,
            reference=lgb_if_ds,
        )

        fold_model = lgb.train(
            lgb_params,
            lgb_if_ds,
            num_boost_round=500,
            valid_sets=[lgb_oof_ds],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )

        oof_probs = fold_model.predict(X_train.iloc[oof_index])
        from sklearn.metrics import roc_auc_score as _auc
        cv_score += _auc(y_train.iloc[oof_index], oof_probs) / N_FOLDS

    return cv_score


In [32]:
# ── Lanzar el estudio Optuna ───────────────────────────────────────────────
study = optuna.create_study(
    direction='maximize',
    storage='sqlite:///../work/db.sqlite3',
    study_name=STUDY_NAME,
    load_if_exists=True,
)

# ── Callback: log concise después de cada trial ───────────────────────────
import time
_start_time = time.time()

def _log_trial(study, trial):
    elapsed = time.time() - _start_time
    best   = study.best_value if study.best_trial else float('nan')
    n_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    flag   = ' ← NEW BEST' if study.best_trial and trial.number == study.best_trial.number else ''
    print(
        f'[Trial {trial.number:3d}] AUC={trial.value:.4f} | '
        f'best={best:.4f} | done={n_done}/{N_TRIALS} | elapsed={elapsed/60:.1f}min{flag}',
        flush=True,
    )

study.optimize(cv_es_lgb_objective, n_trials=N_TRIALS, callbacks=[_log_trial])

print()
print(f'Mejor AUC CV : {study.best_value:.4f}')
print('Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'  {k:25s}: {v}')


[Trial   0] AUC=0.6644 | best=0.6644 | done=1/100 | elapsed=0.1min ← NEW BEST
[Trial   1] AUC=0.6555 | best=0.6644 | done=2/100 | elapsed=0.2min
[Trial   2] AUC=0.6411 | best=0.6644 | done=3/100 | elapsed=0.3min
[Trial   3] AUC=0.6563 | best=0.6644 | done=4/100 | elapsed=0.4min
[Trial   4] AUC=0.6361 | best=0.6644 | done=5/100 | elapsed=0.5min
[Trial   5] AUC=0.6460 | best=0.6644 | done=6/100 | elapsed=0.6min
[Trial   6] AUC=0.6425 | best=0.6644 | done=7/100 | elapsed=0.8min
[Trial   7] AUC=0.6614 | best=0.6644 | done=8/100 | elapsed=0.8min
[Trial   8] AUC=0.6685 | best=0.6685 | done=9/100 | elapsed=0.9min ← NEW BEST
[Trial   9] AUC=0.6350 | best=0.6685 | done=10/100 | elapsed=1.0min
[Trial  10] AUC=0.6471 | best=0.6685 | done=11/100 | elapsed=1.1min
[Trial  11] AUC=0.6605 | best=0.6685 | done=12/100 | elapsed=1.2min
[Trial  12] AUC=0.6658 | best=0.6685 | done=13/100 | elapsed=1.3min
[Trial  13] AUC=0.6720 | best=0.6720 | done=14/100 | elapsed=1.5min ← NEW BEST
[Trial  14] AUC=0.6565 |

In [33]:
# ── Visualización: evolución del QWK CV durante la optimización ────────────
trials_df = study.trials_dataframe()
trials_df_complete = trials_df[trials_df['state'] == 'COMPLETE'].copy()

fig = px.line(
    trials_df_complete,
    x='number',
    y='value',
    title='Evolución del QWK CV durante la Optimización con Optuna (FE pipeline)',
    labels={'number': 'Trial', 'value': 'QWK CV'},
    markers=True,
)
fig.add_hline(
    y=study.best_value,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Mejor: {study.best_value:.4f}',
    annotation_position='bottom right',
)
fig.show()


---
## 9. Entrenamiento Final con K-Fold y Evaluación

Con los mejores hiperparámetros entrenamos un ensemble de 5 modelos (uno por
fold) y promediamos sus predicciones. Evaluamos en **dos conjuntos**:

- `df_meta_train` (20% del original, no visto durante optimización).
- `df_test` (20% restante, también nunca visto).

Cada subset es una estimación independiente de generalización; comparamos contra
el baseline para cuantificar el beneficio del feature engineering.


In [34]:
# ── Entrenamiento final con KFold y mejores hiperparámetros (binary) ───────
best_params = {
    'objective'   : 'binary',
    'metric'      : 'auc',
    'verbosity'   : -1,
    'random_state': SEED,
} | study.best_params

oof_preds_proba  = np.zeros(len(X_train))                   # 1D: P(clase 0)
test_preds_proba = np.zeros(len(X_test))
meta_preds_proba = np.zeros(len(X_meta)) if USE_META_TRAIN else None
best_iterations  = []

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold_idx, (if_index, oof_index) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    print(f'\n─── Fold {fold_idx + 1}/{N_FOLDS} ───────────────────────────────')

    w_if = sample_weight_train[if_index]

    lgb_if_ds = lgb.Dataset(
        data=X_train.iloc[if_index],
        label=y_train.iloc[if_index],
        weight=w_if,
        categorical_feature=CAT_FEATURES,
        free_raw_data=False,
    )
    lgb_oof_ds = lgb.Dataset(
        data=X_train.iloc[oof_index],
        label=y_train.iloc[oof_index],
        categorical_feature=CAT_FEATURES,
        free_raw_data=False,
        reference=lgb_if_ds,
    )

    fold_model = lgb.train(
        best_params,
        lgb_if_ds,
        num_boost_round=500,
        valid_sets=[lgb_oof_ds],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30, verbose=True),
            lgb.log_evaluation(period=50),
        ],
    )

    best_iterations.append(fold_model.best_iteration)

    oof_preds_proba[oof_index] = fold_model.predict(X_train.iloc[oof_index])
    test_preds_proba += fold_model.predict(X_test) / N_FOLDS
    if USE_META_TRAIN:
        meta_preds_proba += fold_model.predict(X_meta) / N_FOLDS

    dump(fold_model, os.path.join(PATH_TO_MODELS, f'lgbm_binary_class0_fold_{fold_idx}.joblib'))

    from sklearn.metrics import roc_auc_score as _auc
    fold_auc = _auc(y_train.iloc[oof_index], oof_preds_proba[oof_index])
    print(f'Fold {fold_idx + 1} AUC OOF: {fold_auc:.4f}  |  Best iter: {fold_model.best_iteration}')

print(f'\nIteraciones por fold: {best_iterations}')
print(f'Iteración promedio  : {np.mean(best_iterations):.0f}')



─── Fold 1/5 ───────────────────────────────
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[17]	valid_0's auc: 0.653955
Fold 1 AUC OOF: 0.6540  |  Best iter: 17

─── Fold 2/5 ───────────────────────────────
Training until validation scores don't improve for 30 rounds
[50]	valid_0's auc: 0.676027
Early stopping, best iteration is:
[37]	valid_0's auc: 0.684465
Fold 2 AUC OOF: 0.6845  |  Best iter: 37

─── Fold 3/5 ───────────────────────────────
Training until validation scores don't improve for 30 rounds
[50]	valid_0's auc: 0.637496
Early stopping, best iteration is:
[31]	valid_0's auc: 0.648649
Fold 3 AUC OOF: 0.6486  |  Best iter: 31

─── Fold 4/5 ───────────────────────────────
Training until validation scores don't improve for 30 rounds
[50]	valid_0's auc: 0.736806
Early stopping, best iteration is:
[46]	valid_0's auc: 0.739671
Fold 4 AUC OOF: 0.7397  |  Best iter: 46

─── Fold 5/5 ───────────────────────────────
Training until vali

In [35]:
# ── Score OOF sobre todo el conjunto de train ──────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score
oof_pred = (oof_preds_proba >= 0.5).astype(int)
auc_oof  = roc_auc_score(y_train, oof_preds_proba)
ap_oof   = average_precision_score(y_train, oof_preds_proba)
f1_oof   = f1_score(y_train, oof_pred)

print(f'AUC OOF (train completo): {auc_oof:.4f}')
print(f'AP OOF (PR-AUC)         : {ap_oof:.4f}')
print(f'F1 OOF (clase 0)        : {f1_oof:.4f}')

plot_confusion_matrix(y_train, oof_pred, title=f'OOF Train Binary — AUC={auc_oof:.4f}').show()


AUC OOF (train completo): 0.6791
AP OOF (PR-AUC)         : 0.0657
F1 OOF (clase 0)        : 0.1034


In [36]:
# ── Evaluación en TEST ─────────────────────────────────────────────────────
y_pred_test = (test_preds_proba >= 0.5).astype(int)

auc_final_test  = roc_auc_score(y_test, test_preds_proba)
ap_final_test   = average_precision_score(y_test, test_preds_proba)
f1_final_test   = f1_score(y_test, y_pred_test)
acc_final_test  = accuracy_score(y_test, y_pred_test)
bacc_final_test = balanced_accuracy_score(y_test, y_pred_test)

if USE_META_TRAIN:
    y_pred_meta = (meta_preds_proba >= 0.5).astype(int)
    auc_final_meta  = roc_auc_score(y_meta, meta_preds_proba)
    ap_final_meta   = average_precision_score(y_meta, meta_preds_proba)
    f1_final_meta   = f1_score(y_meta, y_pred_meta)
    acc_final_meta  = accuracy_score(y_meta, y_pred_meta)
    bacc_final_meta = balanced_accuracy_score(y_meta, y_pred_meta)
else:
    y_pred_meta = auc_final_meta = ap_final_meta = f1_final_meta = acc_final_meta = bacc_final_meta = None

print('=' * 60)
print('RESULTADOS FINALES BINARIOS — TEST')
print(f'  AUC                : {auc_final_test:.4f}')
print(f'  Avg Precision (PR) : {ap_final_test:.4f}')
print(f'  F1 (clase 0)       : {f1_final_test:.4f}')
print(f'  Accuracy           : {acc_final_test:.4f}')
print(f'  Balanced Accuracy  : {bacc_final_test:.4f}')
if USE_META_TRAIN:
    print('-' * 60)
    print('RESULTADOS FINALES BINARIOS — META_TRAIN')
    print(f'  AUC                : {auc_final_meta:.4f}')
    print(f'  Avg Precision (PR) : {ap_final_meta:.4f}')
    print(f'  F1 (clase 0)       : {f1_final_meta:.4f}')
    print(f'  Accuracy           : {acc_final_meta:.4f}')
    print(f'  Balanced Accuracy  : {bacc_final_meta:.4f}')
print('=' * 60)


RESULTADOS FINALES BINARIOS — TEST
  AUC                : 0.6889
  Avg Precision (PR) : 0.1332
  F1 (clase 0)       : 0.1560
  Accuracy           : 0.9603
  Balanced Accuracy  : 0.5588


In [37]:
# ── Comparación contra Baseline ────────────────────────────────────────────
metrics_test = pd.DataFrame({
    'Métrica'      : ['AUC', 'Avg Precision', 'F1 (clase 0)', 'Accuracy', 'Balanced Accuracy'],
    'Baseline_Test': [auc_baseline_test, ap_baseline_test, f1_baseline_test, acc_baseline_test, bacc_baseline_test],
    'FE_Test'      : [auc_final_test, ap_final_test, f1_final_test, acc_final_test, bacc_final_test],
    'Δ_Test'       : [auc_final_test  - auc_baseline_test,
                      ap_final_test   - ap_baseline_test,
                      f1_final_test   - f1_baseline_test,
                      acc_final_test  - acc_baseline_test,
                      bacc_final_test - bacc_baseline_test],
})

if USE_META_TRAIN:
    metrics_test['Baseline_Meta'] = [auc_baseline_meta, ap_baseline_meta, f1_baseline_meta, acc_baseline_meta, bacc_baseline_meta]
    metrics_test['FE_Meta']       = [auc_final_meta, ap_final_meta, f1_final_meta, acc_final_meta, bacc_final_meta]
    metrics_test['Δ_Meta']        = [auc_final_meta  - auc_baseline_meta,
                                      ap_final_meta   - ap_baseline_meta,
                                      f1_final_meta   - f1_baseline_meta,
                                      acc_final_meta  - acc_baseline_meta,
                                      bacc_final_meta - bacc_baseline_meta]

display(metrics_test.round(4))

plot_confusion_matrix(
    y_test, y_pred_test,
    title=f'Test Final Binary (Ensemble {N_FOLDS} Folds) — AUC={auc_final_test:.4f}',
).show()

if USE_META_TRAIN:
    plot_confusion_matrix(
        y_meta, y_pred_meta,
        title=f'Meta-Train Binary (Ensemble {N_FOLDS} Folds) — AUC={auc_final_meta:.4f}',
    ).show()


,Métrica,Baseline_Test,FE_Test,Δ_Test
0,AUC,0.6459,0.6889,0.0430
1,Avg Precision,0.1032,0.1332,0.0300
2,F1 (clase 0),0.0909,0.1560,0.0651
3,Accuracy,0.9733,0.9603,-0.0130
4,Balanced Accuracy,0.5240,0.5588,0.0348


In [38]:
# ── Reporte por clase (TEST) — binario ─────────────────────────────────────
report_test = classification_report(y_test, y_pred_test, output_dict=True,
                                     target_names=['no_class_0', 'class_0'])
report_test_df = (
    pd.DataFrame(report_test)
    .T
    .iloc[:2][['precision', 'recall', 'f1-score', 'support']]
)
print('Reporte TEST:')
display(report_test_df.round(3))

fig = px.bar(
    report_test_df.reset_index().rename(columns={'index': 'Clase'}),
    x='Clase', y='f1-score', text='f1-score',
    title='F1-Score por Clase — TEST (binario clase 0)',
    color='f1-score', color_continuous_scale='Blues',
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(coloraxis_showscale=False, yaxis_range=[0, 1])
fig.show()


Reporte TEST:


,precision,recall,f1-score,support
no_class_0,0.976,0.984,0.980,2917.0
class_0,0.186,0.134,0.156,82.0


---
## 10. Análisis de Features Importantes

Dos análisis complementarios:

**(a) Importancia agregada del modelo final**
- Promedio de `feature_importance(gain)` y `feature_importance(split)` sobre los
  5 folds.
- Top 20 individual + agregación por **grupo de FE** (Descripción, Rescuista,
  Geo, Media, Fee, Breed, Edad, Salud, Interacciones, Cantidad).

**(b) Análisis incremental "base + grupo"**
- Entrenamos modelos LGBM (parámetros fijos, sin Optuna) sobre:
  - solo `base` (features que no pertenecen a ningún grupo de FE),
  - `base + cada grupo` (uno a la vez).
- Reportamos el **ΔQWK** en TEST de cada modelo respecto del modelo `base`.
- Esto permite ver qué grupos aportan información incremental real.


In [39]:
# ── Importancia agregada (Gain / Split) ────────────────────────────────────
importance_gain  = np.zeros(len(features_fe))
importance_split = np.zeros(len(features_fe))

for fold_idx in range(N_FOLDS):
    fold_model = load(os.path.join(PATH_TO_MODELS, f'lgbm_binary_class0_fold_{fold_idx}.joblib'))
    importance_gain  += fold_model.feature_importance(importance_type='gain')  / N_FOLDS
    importance_split += fold_model.feature_importance(importance_type='split') / N_FOLDS

importance_df = pd.DataFrame({
    'feature'         : features_fe,
    'importance_gain' : importance_gain,
    'importance_split': importance_split,
}).sort_values('importance_gain', ascending=False).reset_index(drop=True)

print('Top 15 features por Gain:')
display(importance_df.head(15))


Top 15 features por Gain:


,feature,importance_gain,importance_split
0,Breed1_enc,6128.173332,35.8
1,rescuer_listing_count,2381.945785,21.4
2,rescuer_vol_x_photo,2214.609837,26.6
3,desc_sentiment_english_score,2009.257004,35.6
4,desc_pca_8,1943.807087,37.2
5,desc_pca_19,1761.271974,30.6
6,desc_pca_1,1688.427442,30.4
7,desc_pca_17,1687.444629,30.8
8,low_vol_high_desc,1686.073555,26.6
9,desc_pca_22,1640.059143,30.2


In [40]:
# ── Top 20 por Gain ────────────────────────────────────────────────────────
top20_gain = importance_df.head(20).sort_values('importance_gain', ascending=True)
fig = go.Figure(go.Bar(
    y=top20_gain['feature'], x=top20_gain['importance_gain'],
    orientation='h', marker_color='steelblue',
))
fig.update_layout(
    title='Top 20 Features — Importancia por Gain (promedio 5 folds)',
    xaxis_title='Importancia (Gain)', yaxis_title='Feature', height=600,
)
fig.show()

# ── Top 20 por Split ───────────────────────────────────────────────────────
top20_split = (
    importance_df.sort_values('importance_split', ascending=False)
    .head(20)
    .sort_values('importance_split', ascending=True)
)
fig = go.Figure(go.Bar(
    y=top20_split['feature'], x=top20_split['importance_split'],
    orientation='h', marker_color='coral',
))
fig.update_layout(
    title='Top 20 Features — Importancia por Split Count (promedio 5 folds)',
    xaxis_title='Importancia (Split Count)', yaxis_title='Feature', height=600,
)
fig.show()


In [41]:
# ── Definición de grupos de features ───────────────────────────────────────
# Patrones consistentes con los nombres generados por las funciones add_*.
# La función helper `classify_features_into_groups` también se usa en el
# análisis incremental "base + grupo" más abajo.

def classify_features_into_groups(feat_list: list[str]) -> dict[str, list[str]]:
    groups = {
        'Descripción (NLP)' : [f for f in feat_list if f.startswith('desc_') or 'sentiment' in f or 'text_' in f or 'urgency' in f],
        'Rescuista'         : [f for f in feat_list if f.startswith('rescuer_') or 'rescuer' in f.lower()],
        'Geografía'         : [f for f in feat_list if f.startswith('state_') or 'population' in f or 'gdp' in f or 'urban' in f or 'area_km2' in f],
        'Media (Foto/Video)': [f for f in feat_list if 'photo' in f.lower() or 'video' in f.lower() or 'media' in f.lower()],
        'Fee / Precio'      : [f for f in feat_list if 'fee' in f.lower()],
        'Breed / Raza'      : [f for f in feat_list if 'breed' in f.lower() or 'mixed' in f.lower()],
        'Edad'              : [f for f in feat_list if 'age' in f.lower() or f == 'is_young'],
        'Salud'             : [f for f in feat_list if f in ['Health', 'health_score', 'vaccinated_bin', 'dewormed_bin', 'sterilized_bin']
                                                    or f.startswith('health_')],
        'Interacciones'     : [f for f in feat_list if '_x_' in f or 'interaction' in f.lower()],
        'Cantidad'          : [f for f in feat_list if 'quantity' in f.lower() or f == 'is_single'],
        'Color'             : [f for f in feat_list if 'color' in f.lower() and f != 'has_color'],
    }
    classified = {f for grp in groups.values() for f in grp}
    groups['Base / Otros'] = [f for f in feat_list if f not in classified]
    return groups

feature_groups = classify_features_into_groups(features_fe)

print('Conteo de features por grupo:')
for g, fs in feature_groups.items():
    print(f'  {g:25s}: {len(fs):3d}')


Conteo de features por grupo:
  Descripción (NLP)        :  38
  Rescuista                :   9
  Geografía                :   7
  Media (Foto/Video)       :  12
  Fee / Precio             :   7
  Breed / Raza             :   4
  Edad                     :   6
  Salud                    :   8
  Interacciones            :  14
  Cantidad                 :   4
  Color                    :   1
  Base / Otros             :  17


In [42]:
# ── Importancia agregada por grupo (Gain) ──────────────────────────────────
group_importance = {}
for group_name, group_feats in feature_groups.items():
    mask = importance_df['feature'].isin(group_feats)
    group_importance[group_name] = importance_df.loc[mask, 'importance_gain'].sum()

group_df = (
    pd.DataFrame(list(group_importance.items()), columns=['Grupo', 'Importance_Gain'])
    .query('Importance_Gain > 0')
    .sort_values('Importance_Gain', ascending=False)
    .reset_index(drop=True)
)

fig = px.pie(
    group_df, names='Grupo', values='Importance_Gain',
    title='Importancia por Grupo de Features (Gain Total)', hole=0.35,
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

display(group_df)


,Grupo,Importance_Gain
0,Descripción (NLP),41225.705307
1,Rescuista,10194.705423
2,Breed / Raza,7123.426319
3,Interacciones,6895.351828
4,Media (Foto/Video),6881.394036
5,Base / Otros,3270.424574
6,Edad,2006.571113
7,Geografía,1428.214144
8,Fee / Precio,1083.558978
9,Salud,881.518830


### 10.b Análisis incremental: `base + grupo`

Para cuantificar **cuánto suma cada grupo de features individualmente**,
entrenamos un modelo LGBM con parámetros fijos (los mejores de Optuna, sin
re-optimizar) usando:

- solo `base` (features que no pertenecen a ningún grupo FE específico),
- `base + grupo_k` para cada grupo `k`.

Reportamos QWK en TEST de cada variante y el ΔQWK respecto de `base`.

**Importante:** este análisis es indicativo, no exhaustivo — el orden y las
interacciones entre grupos pueden hacer que la suma de los aportes individuales
no coincida con el aporte conjunto.


In [43]:
# ── Helper: entrena un modelo simple con un subset de features ─────────────
def train_eval_subset(feat_subset: list[str], label: str = '') -> dict:
    """Entrena un único LGBM binary con los mejores params, sin Optuna,
    usando el subset de features dado. Devuelve métricas en TEST.
    """
    feat_subset = [f for f in feat_subset if f in X_train.columns]
    if len(feat_subset) == 0:
        return {'label': label, 'n_feats': 0, 'auc_test': np.nan, 'f1_test': np.nan}

    cat_in_subset = [c for c in CAT_FEATURES if c in feat_subset]

    ds = lgb.Dataset(
        data=X_train[feat_subset],
        label=y_train,
        weight=sample_weight_train,
        categorical_feature=cat_in_subset,
        free_raw_data=False,
    )

    model = lgb.train(
        best_params,
        ds,
        num_boost_round=int(np.mean(best_iterations)) if best_iterations else 200,
    )

    probs = model.predict(X_test[feat_subset])
    y_hat = (probs >= 0.5).astype(int)
    return {
        'label'    : label,
        'n_feats'  : len(feat_subset),
        'auc_test' : roc_auc_score(y_test, probs),
        'f1_test'  : f1_score(y_test, y_hat),
    }


In [44]:
# ── Ejecución: base, base+grupo (uno a la vez), todo ───────────────────────
base_feats = feature_groups['Base / Otros']

results = []
results.append(train_eval_subset(base_feats, label='base'))

for gname, gfeats in feature_groups.items():
    if gname == 'Base / Otros' or len(gfeats) == 0:
        continue
    combined = list(set(base_feats) | set(gfeats))
    results.append(train_eval_subset(combined, label=f'base + {gname}'))

results.append(train_eval_subset(features_fe, label='ALL features'))

incremental_df = pd.DataFrame(results)
auc_base = incremental_df.loc[incremental_df['label'] == 'base', 'auc_test'].values[0]
incremental_df['Δ_AUC_vs_base'] = incremental_df['auc_test'] - auc_base
incremental_df = incremental_df.sort_values('auc_test', ascending=False).reset_index(drop=True)

print('Análisis incremental binario — AUC en TEST por configuración de features:')
display(incremental_df.round(4))


Análisis incremental binario — AUC en TEST por configuración de features:


,label,n_feats,auc_test,f1_test,Δ_AUC_vs_base
0,ALL features,109,0.6844,0.1552,0.0568
1,base + Rescuista,26,0.6806,0.1257,0.0529
2,base + Breed / Raza,21,0.6513,0.0856,0.0237
3,base + Descripción (NLP),55,0.6505,0.1189,0.0229
4,base + Fee / Precio,24,0.6485,0.0992,0.0209
5,base + Geografía,24,0.6448,0.0930,0.0171
6,base + Color,18,0.6349,0.1073,0.0072
7,base + Salud,25,0.6330,0.0939,0.0053
8,base + Interacciones,31,0.6282,0.0995,0.0006
9,base,17,0.6277,0.1065,0.0000


In [45]:
# ── Visualización del aporte incremental por grupo (ΔAUC) ──────────────────
plot_df = incremental_df[~incremental_df['label'].isin(['base', 'ALL features'])].copy()
plot_df = plot_df.sort_values('Δ_AUC_vs_base', ascending=True)

fig = go.Figure(go.Bar(
    y=plot_df['label'],
    x=plot_df['Δ_AUC_vs_base'],
    orientation='h',
    marker_color=['#2ecc71' if v > 0 else '#e74c3c' for v in plot_df['Δ_AUC_vs_base']],
    text=plot_df['Δ_AUC_vs_base'].round(4),
    textposition='outside',
))
fig.update_layout(
    title=f'Aporte Incremental por Grupo (ΔAUC vs base — AUC base = {auc_base:.4f})',
    xaxis_title='ΔAUC (TEST)', yaxis_title='Configuración',
    height=500,
)
fig.add_vline(x=0, line_dash='dash', line_color='black', opacity=0.5)
fig.show()


---
## 11. Exportar predicciones para metamodelo

Generamos archivos CSV con las **probabilidades por clase** que predijo este
modelo, listos para ser consumidos por un metamodelo posterior. Cada archivo
contiene una fila por `PetID` con 5 columnas de probabilidad (clases 0 a 4).

**Importante**:

- Para `train` exportamos las **predicciones OOF** (`oof_preds_proba`), no las
  predicciones del modelo entrenado sobre todo el train. Son las únicas predicciones
  *unbiased* sobre los pets de train (cada uno fue predicho por un modelo que NO lo
  vio durante el entrenamiento del fold).
- Para `meta_train` y `test` exportamos el **ensemble** (promedio) de los 5
  modelos de fold (que es la predicción "oficial" del modelo).
- La ruta es **relativa** al notebook (`../output/predictions/`) para que el
  artefacto pueda mudarse a otro repo sin cambios.


In [46]:
# ── Exportación de predicciones para metamodelo ───────────────────────────
# El metamodelo (regresión logística) espera 5 columnas de probabilidad por
# pet. Como este modelo es binario, distribuimos la masa así:
#   prob_class_0     = p                ← probabilidad real predicha
#   prob_class_1..4  = (1 - p) / 4      ← uniforme sobre las clases no-0
# Resultado: distribución válida (suma=1). La única columna informativa real
# es prob_class_0; las otras 4 son función lineal de la primera y el meta-
# modelo las maneja con regularización L2.
import os

PRED_OUTPUT_DIR = '../output/predictions/'
os.makedirs(PRED_OUTPUT_DIR, exist_ok=True)

MODEL_NAME = 'lgbm_binary_class0'

PROB_COLS = [f'prob_class_{i}' for i in range(NUM_CLASSES)]


def _build_pred_df(petids, probs_class0):
    """Convierte un array 1D de P(clase 0) en DataFrame con 5 columnas."""
    p = np.asarray(probs_class0).reshape(-1)
    rest = (1.0 - p) / (NUM_CLASSES - 1)

    full_probs = np.full((len(p), NUM_CLASSES), 0.0)
    full_probs[:, 0] = p
    for k in range(1, NUM_CLASSES):
        full_probs[:, k] = rest

    df = pd.DataFrame(full_probs, columns=PROB_COLS)
    df.insert(0, 'PetID', petids)
    return df


# ── TRAIN: predicciones OOF ────────────────────────────────────────────────
train_pred_df = _build_pred_df(
    petids=df_train_fe['PetID'].values,
    probs_class0=oof_preds_proba,
)
train_path = os.path.join(PRED_OUTPUT_DIR, f'{MODEL_NAME}_train_preds.csv')
train_pred_df.to_csv(train_path, index=False)
print(f'  Guardado: {train_path}  shape={train_pred_df.shape}')

# ── TEST ──────────────────────────────────────────────────────────────────
test_pred_df = _build_pred_df(
    petids=df_test_fe['PetID'].values,
    probs_class0=test_preds_proba,
)
test_path = os.path.join(PRED_OUTPUT_DIR, f'{MODEL_NAME}_test_preds.csv')
test_pred_df.to_csv(test_path, index=False)
print(f'  Guardado: {test_path}  shape={test_pred_df.shape}')

# ── META_TRAIN ────────────────────────────────────────────────────────────
if USE_META_TRAIN:
    meta_pred_df = _build_pred_df(
        petids=df_meta_train_fe['PetID'].values,
        probs_class0=meta_preds_proba,
    )
    meta_path = os.path.join(PRED_OUTPUT_DIR, f'{MODEL_NAME}_meta_preds.csv')
    meta_pred_df.to_csv(meta_path, index=False)
    print(f'  Guardado: {meta_path}  shape={meta_pred_df.shape}')
else:
    print(f'  META_TRAIN no aplica (USE_META_TRAIN=False) — no se exporta meta_preds.')

# ── Validaciones ──────────────────────────────────────────────────────────
print('\nValidaciones:')
print(f'  Train probs suman ~1: {np.allclose(train_pred_df[PROB_COLS].sum(axis=1), 1.0, atol=1e-6)}')
print(f'  Test  probs suman ~1: {np.allclose(test_pred_df[PROB_COLS].sum(axis=1), 1.0, atol=1e-6)}')
if USE_META_TRAIN:
    print(f'  Meta  probs suman ~1: {np.allclose(meta_pred_df[PROB_COLS].sum(axis=1), 1.0, atol=1e-6)}')

print(f'\n  PetIDs unicos en train: {train_pred_df["PetID"].nunique()} / {len(train_pred_df)}')
print(f'  PetIDs unicos en test : {test_pred_df["PetID"].nunique()} / {len(test_pred_df)}')
if USE_META_TRAIN:
    print(f'  PetIDs unicos en meta : {meta_pred_df["PetID"].nunique()} / {len(meta_pred_df)}')

# ── Estadisticas de la columna informativa (prob_class_0) ─────────────────
print('\nEstadisticas prob_class_0 (la unica columna realmente informativa):')
print(f'  Train: mean={oof_preds_proba.mean():.4f}, std={oof_preds_proba.std():.4f}, p99={np.percentile(oof_preds_proba, 99):.4f}')
print(f'  Test : mean={test_preds_proba.mean():.4f}, std={test_preds_proba.std():.4f}, p99={np.percentile(test_preds_proba, 99):.4f}')

print('\nPreview train_pred_df (primeras 3 filas):')
display(train_pred_df.head(3))


  Guardado: ../output/predictions/lgbm_binary_class0_train_preds.csv  shape=(11994, 6)
  Guardado: ../output/predictions/lgbm_binary_class0_test_preds.csv  shape=(2999, 6)
  META_TRAIN no aplica (USE_META_TRAIN=False) — no se exporta meta_preds.

Validaciones:
  Train probs suman ~1: True
  Test  probs suman ~1: True

  PetIDs unicos en train: 11994 / 11994
  PetIDs unicos en test : 2999 / 2999

Estadisticas prob_class_0 (la unica columna realmente informativa):
  Train: mean=0.4308, std=0.0383, p99=0.5184
  Test : mean=0.4340, std=0.0305, p99=0.5078

Preview train_pred_df (primeras 3 filas):


,PetID,prob_class_0,prob_class_1,prob_class_2,prob_class_3,prob_class_4
0,23b64fe21,0.430776,0.142306,0.142306,0.142306,0.142306
1,6e09bfe1f,0.392862,0.151785,0.151785,0.151785,0.151785
2,48a44eac5,0.432647,0.141838,0.141838,0.141838,0.141838


---
## 12. Conclusiones

- **Baseline vs FE:** la tabla de la sección 9 cuantifica el aporte agregado del
  pipeline de feature engineering frente al modelo crudo.
- **Top features:** el análisis de Gain identifica los predictores individuales
  más relevantes; suelen mezclar features base (Age, Breed) con features
  derivadas (target encoding, NLP, rescuer aggregations).
- **Grupos:** la torta de importancia agregada y el análisis incremental
  (`base + grupo`) son complementarios:
  - La torta dice "del modelo final, qué fracción del Gain se la lleva cada
    grupo".
  - El incremental dice "si parto de base, qué grupo me da el mayor ΔQWK por sí
    solo".
  Diferencias entre ambos análisis suelen indicar interacciones entre grupos.
